# Devnagri LLM — Clean Kaggle Pipeline (Hindi Only)

In [1]:
from pathlib import Path
for p in Path("/kaggle/input").iterdir():
    print(p)
    for sub in p.rglob("persistent"):
        print("  found:", sub)

/kaggle/input/datasets
  found: /kaggle/input/datasets/parthbramhecha007/hindi-checkpoint/persistent


In [2]:
# ============================================================
# 0. KAGGLE / RUNTIME CONFIGURATION
# ============================================================
import os
import sys
import time
from pathlib import Path

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")

REPO_DIR = Path("/kaggle/working/Devnagri_LLM")
PERSISTENT_DIR = Path("/kaggle/working/persistent")
SPLIT_SOURCE = Path(
    "/kaggle/input/datasets/parthbramhecha007/"
    "split-devnagri-compression/splits"
)

# Hindi-only run. (Full pipeline normally loops over
# pipeline.config.LANGUAGES == ["hindi", "marathi", "sanskrit"];
# this notebook restricts every stage below to hindi.)
LANGUAGES = ["hindi"]

os.chdir("/kaggle/working")
PERSISTENT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Session wall-clock budget.
#
# Kaggle kills the *whole kernel* at a hard 12h GPU-session limit --
# not a catchable Python exception, just a SIGKILL. Every later stage
# in this notebook (Stage 2d's resumable rounds, the Stage 2d guard,
# Stage 3) must measure itself against THIS clock, not against its
# own internal --max-hours budget, or a stage can start with (say)
# 20 minutes of real time left and get killed mid-run with no
# checkpoint, no traceback, and no results file.
# ------------------------------------------------------------------
SESSION_START = time.time()
KAGGLE_SESSION_LIMIT_HOURS = 12.0   # Kaggle's hard wall-clock cap for this session
SESSION_SAFETY_BUFFER_HOURS = 2.5   # reserved for Stage 3 model load + 3-condition compression + Stage 4
                                     # (bumped from 1.5 -- gives the notebook more room to finish
                                     # its LAST cell and commit as "Complete" before Kaggle's hard
                                     # 12h kill, since a killed session may not leave a resumable
                                     # Output the way a normal "Complete" commit does)

def hours_elapsed():
    """Wall-clock hours since this notebook/session started."""
    return (time.time() - SESSION_START) / 3600.0

def hours_remaining_in_session():
    """Hours left before Kaggle's hard 12h cap, minus the safety buffer."""
    return KAGGLE_SESSION_LIMIT_HOURS - SESSION_SAFETY_BUFFER_HOURS - hours_elapsed()

print("=" * 72)
print("DEVNAGRI LLM — CLEAN KAGGLE PIPELINE (HINDI ONLY)")
print("=" * 72)
print("Python:", sys.version)
print("Working directory:", Path.cwd())
print("Repository:", REPO_DIR)
print("Dataset source:", SPLIT_SOURCE)
print("Languages:", ", ".join(LANGUAGES))
print(f"Session budget: {KAGGLE_SESSION_LIMIT_HOURS:.1f}h hard cap, "
      f"{SESSION_SAFETY_BUFFER_HOURS:.1f}h reserved for Stage 3/4 "
      f"-> {hours_remaining_in_session():.2f}h available for Stage 2d rounds.")


DEVNAGRI LLM — CLEAN KAGGLE PIPELINE (HINDI ONLY)
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Working directory: /kaggle/working
Repository: /kaggle/working/Devnagri_LLM
Dataset source: /kaggle/input/datasets/parthbramhecha007/split-devnagri-compression/splits
Languages: hindi
Session budget: 12.0h hard cap, 2.5h reserved for Stage 3/4 -> 9.50h available for Stage 2d rounds.


In [3]:
# ============================================================
# 0b. RESTORE CHECKPOINT STATE FROM A PREVIOUS SESSION (IF ATTACHED)
# ============================================================
import shutil
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

def find_prior_persistent_dir():
    """Locate a 'persistent' directory inside any attached input dataset.

    This is how a PREVIOUS committed version of THIS SAME notebook's Output
    gets found, once it is attached as an input to the current version:
    Notebook -> Add Input -> Notebook Output Files -> this notebook,
    latest version. Kaggle mounts it read-only under /kaggle/input/...,
    but the exact depth varies by input type:
      - a flat "Dataset" input mounts at    /kaggle/input/<slug>/...
      - a "Notebook Output Files" input mounts nested under
        /kaggle/input/notebooks/<username>/<notebook-slug>/...
    so we search recursively (rglob) rather than assuming a fixed depth,
    and pick the shallowest match in case more than one 'persistent'
    directory happens to be attached.
    """
    if not INPUT_ROOT.exists():
        return None
    matches = [p for p in INPUT_ROOT.rglob("persistent") if p.is_dir()]
    if not matches:
        return None
    matches.sort(key=lambda p: len(p.parts))
    return matches[0]

prior_persistent = find_prior_persistent_dir()

# FIX: the old code did
#     shutil.copytree(prior_persistent, PERSISTENT_DIR, dirs_exist_ok=True)
# unconditionally -- copying EVERY language's Stage 2d checkpoints and
# Stage 3 results into /kaggle/working regardless of whether this
# session's LANGUAGES actually touches them. For a hindi-only run,
# marathi's and sanskrit's full Stage 3 compression results (never read
# this session) were restored before a single byte of this session's own
# work happened, consuming most of the fixed 20GB /kaggle/working quota
# and leaving seed_2 only ~3.90GB free -- not even enough for its first
# periodic checkpoint (crashed at step 200 with "No space left on
# device").
#
# Restore only this session's LANGUAGES up front. Nothing is lost:
# RECOMBINE_LATER records what was skipped, and the final cell in this
# notebook copies it back in from the (still-mounted, read-only) input
# dir right before the session ends -- so it's still part of this run's
# committed Output, it just doesn't occupy disk during the GPU-heavy
# training in between, which is when the room is actually needed.
RECOMBINE_LATER = []

if prior_persistent is not None:
    print(f"↺ Found checkpoint state from a previous session: {prior_persistent}")
    PERSISTENT_DIR.mkdir(parents=True, exist_ok=True)

    for kind in ("models/devaware_finetuned", "results"):
        src_root = prior_persistent / kind
        if not src_root.exists():
            continue
        for lang_dir in src_root.iterdir():
            if not lang_dir.is_dir():
                continue
            dest = PERSISTENT_DIR / kind / lang_dir.name
            if lang_dir.name in LANGUAGES:
                shutil.copytree(lang_dir, dest, dirs_exist_ok=True)
            else:
                RECOMBINE_LATER.append((lang_dir, dest))

    # tokenizer/, logs/, eval/ are small and shared, not per-language --
    # restore them in full as before.
    for kind in ("tokenizer", "logs", "eval"):
        src = prior_persistent / kind
        if src.exists():
            shutil.copytree(src, PERSISTENT_DIR / kind, dirs_exist_ok=True)

    restored_finals = sorted(
        p.parent.name for p in PERSISTENT_DIR.glob("models/devaware_finetuned/*/final")
    )
    restored_steps = sorted(
        f"{p.parent.name}/{p.name}"
        for p in PERSISTENT_DIR.glob("models/devaware_finetuned/*/step_*")
    )
    restored_results = sorted(
        p.parent.name for p in PERSISTENT_DIR.glob("results/*/compression_results.json")
    )
    print(f"✓ Restored persistent state into {PERSISTENT_DIR} (this session's languages: {LANGUAGES})")
    print(f"  Finished Stage 2d checkpoints found:    {restored_finals or 'none'}")
    print(f"  In-progress Stage 2d checkpoints found: {restored_steps or 'none'}")
    print(f"  Finished Stage 3 results found:         {restored_results or 'none'}")
    if RECOMBINE_LATER:
        deferred = sorted(f"{s.parent.name}/{s.name}" for s, _ in RECOMBINE_LATER)
        print(f"  Deferred (not this session's languages -- recombined before commit): {deferred}")

    free_gb = shutil.disk_usage("/kaggle/working").free / (1024 ** 3)
    print(f"  {free_gb:.2f} GB free on /kaggle/working after restore")
else:
    print(
        "No previous session's Output is attached as an input yet -- "
        "starting fresh. After THIS run finishes and you commit it, attach "
        "this notebook's own Output as an input on the NEXT version "
        "(Add Input -> Notebook Output Files -> this notebook, latest "
        "version) so its checkpoints carry forward. See the documentation "
        "for the exact steps."
    )


↺ Found checkpoint state from a previous session: /kaggle/input/datasets/parthbramhecha007/hindi-checkpoint/persistent
✓ Restored persistent state into /kaggle/working/persistent (this session's languages: ['hindi'])
  Finished Stage 2d checkpoints found:    ['hindi']
  In-progress Stage 2d checkpoints found: none
  Finished Stage 3 results found:         ['hindi']
  Deferred (not this session's languages -- recombined before commit): ['results/marathi', 'results/sanskrit']
  3.90 GB free on /kaggle/working after restore


## Cross-session checkpoint restore

Kaggle wipes `/kaggle/working` between separate sessions -- only a **committed**
version's final `/kaggle/working` state is saved, as that version's "Output".
The cell below looks for a *previous* committed version of this notebook
attached as an input (see the documentation for the exact steps) and copies
its `persistent/` checkpoint tree back in before anything else runs, so
Stage 2d resumes instead of restarting from step 0.

In [4]:
# ============================================================
# 0b (duplicate, neutralized) -- see cell 0b above.
# ============================================================
# This cell used to re-run the same "restore persistent state" logic a
# second time with a simpler (non-recursive) glob, redundantly re-copying
# everything cell 3 already restored. Harmless but wasteful, and it does
# NOT know about RECOMBINE_LATER from the cell above, so leaving it active
# would silently copy back in the languages cell 3 just deferred --
# defeating the whole point of the fix. Left as a no-op rather than
# deleted so cell numbering elsewhere in the notebook doesn't shift.
print("(skipped -- duplicate of cell 0b, see above)")


(skipped -- duplicate of cell 0b, see above)


In [5]:
# ============================================================
# 1. REPOSITORY + DATASET — NON-DESTRUCTIVE
# ============================================================
import subprocess
import shutil

REPO_URL = "https://github.com/PARTH-BRAMHECHA/Devnagri_LLM.git"

# Never remove an existing repository. This preserves checkpoints and edits.
if REPO_DIR.exists():
    print("✓ Repository path already exists; reusing it.")
else:
    print("Repository not found; attempting clone...")
    result = subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        text=True,
    )
    if result.returncode != 0 or not REPO_DIR.exists():
        raise RuntimeError(
            "Could not clone Devnagri_LLM. "
            "Enable Kaggle Internet or attach/upload the repository."
        )

os.chdir(REPO_DIR)

git = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"],
    capture_output=True, text=True
)
print("✓ Repository:", REPO_DIR)
if git.returncode == 0:
    print("✓ Commit:", git.stdout.strip())

# Dataset link: create only if absent. Never delete a real directory.
data_dir = REPO_DIR / "data"
data_dir.mkdir(parents=True, exist_ok=True)
split_link = data_dir / "splits"

if split_link.is_symlink():
    if split_link.resolve() != SPLIT_SOURCE.resolve():
        split_link.unlink()
        split_link.symlink_to(SPLIT_SOURCE, target_is_directory=True)
elif split_link.exists():
    print("✓ data/splits already exists; leaving it untouched.")
else:
    if not SPLIT_SOURCE.exists():
        raise FileNotFoundError(
            f"Kaggle dataset was not found at {SPLIT_SOURCE}. "
            "Attach the split-devnagri-compression dataset."
        )
    split_link.symlink_to(SPLIT_SOURCE, target_is_directory=True)

print("Dataset path:", split_link)
print("Dataset resolves to:", split_link.resolve())


Repository not found; attempting clone...


Cloning into '/kaggle/working/Devnagri_LLM'...


✓ Repository: /kaggle/working/Devnagri_LLM
✓ Commit: 4f4c59e
Dataset path: /kaggle/working/Devnagri_LLM/data/splits
Dataset resolves to: /kaggle/input/datasets/parthbramhecha007/split-devnagri-compression/splits


In [6]:
# ============================================================
# 1b. PATCH: fix disk-full crash in Stage 2d checkpoint saving
# ============================================================
# ROOT CAUSE (confirmed against the actual repo source): the live
# stage2d_vocab_extend.py already prunes the OLD checkpoint before writing
# the NEW one -- that part was fine. The real bug is that its cleanup-on-
# failure handler only did `except OSError`. safetensors raises
# safetensors._safetensors_rust.SafetensorError for "No space left on
# device" (os error 28), which is NOT a subclass of OSError, so that
# handler never fired for the exact crash it exists to catch -- the error
# propagated uncaught instead of getting cleaned up and re-raised with a
# clear message.
#
# A SECOND bug made this invisible: this cell's own "already patched?"
# detection looked for the string "shutil.rmtree(out_dir, ignore_errors=True)"
# to decide whether to skip patching -- but that string is present in BOTH
# the old OSError-only version and the fixed version, so this cell was
# silently skipping the real fix, believing it already applied.
#
# There is ALSO a cross-seed disk bug: _save_checkpoint only prunes
# step_N/ dirs inside its OWN save_dir (e.g. only seed_1/), never sibling
# seed dirs (e.g. seed_0/) sharing the same 20GB Kaggle quota. If seed_0
# stops before reaching `final`, its last step_N/ checkpoint (several GB)
# is never cleaned up by anything and can starve seed_1's write of space.
# This patch fixes all three issues and re-applies every session so it
# self-heals even before the fixes are pushed to GitHub.
import shutil
from pathlib import Path

_target = REPO_DIR / "pipeline" / "stage2d_vocab_extend.py"
_src = _target.read_text(encoding="utf-8")

_OLD = '''    except OSError as e:
        # Most likely still ENOSPC (e.g. pre-prune wasn't enough headroom,
        # or something else on disk grew mid-run). Clean up the partial
        # write so a half-finished checkpoint can never be mistaken for a
        # resumable one by _find_latest_checkpoint, then re-raise so the
        # caller's existing OOM/wall-clock handling still applies.
        shutil.rmtree(out_dir, ignore_errors=True)
        free_gb = shutil.disk_usage(save_dir).free / (1024 ** 3)
        print(f"  ✗ Failed to save checkpoint at step {step}: {e} "
              f"({free_gb:.2f} GB free on {save_dir}). Removed partial "
              f"write at {out_dir}.")
        raise'''

_NEW = '''    except Exception as e:
        # FIX: was `except OSError` only, which never caught
        # safetensors._safetensors_rust.SafetensorError (the actual error
        # class for disk-full during model.save_pretrained). Catching
        # Exception broadly, with an unconditional re-raise, makes this a
        # catch-all cleanup step instead of a silent miss.
        shutil.rmtree(out_dir, ignore_errors=True)
        free_gb = shutil.disk_usage(save_dir).free / (1024 ** 3)
        print(f"  ✗ Failed to save checkpoint at step {step}: "
              f"{type(e).__name__}: {e} "
              f"({free_gb:.2f} GB free on {save_dir}). Removed partial "
              f"write at {out_dir}.")
        raise'''

_CROSS_SEED_PATCH_MARKER = "_prune_incomplete_sibling_dirs"

_CROSS_SEED_FN = '''

def _prune_incomplete_sibling_dirs(save_dir: Path, min_free_gb: float = 8.0):
    """Free disk from OTHER seed/lang checkpoint trees sharing this quota,
    if free space is tight. See patch cell 1b for full rationale."""
    devaware_root = save_dir.parents[1] if save_dir.parent.name.startswith("seed_") \\
        else save_dir.parent
    if not devaware_root.exists():
        return
    free_gb = shutil.disk_usage(devaware_root).free / (1024 ** 3)
    if free_gb >= min_free_gb:
        return
    print(f"  ⚠ Only {free_gb:.2f} GB free -- scanning sibling checkpoint "
          f"dirs under {devaware_root} for prunable step_N/ dirs...")
    for lang_dir in devaware_root.iterdir():
        if not lang_dir.is_dir():
            continue
        for candidate_dir in [lang_dir, *[d for d in lang_dir.glob("seed_*") if d.is_dir()]]:
            if candidate_dir == save_dir:
                continue
            for step_dir in sorted(candidate_dir.glob("step_*")):
                print(f"  [cross-seed prune] {step_dir} (freeing space for {save_dir})")
                shutil.rmtree(step_dir, ignore_errors=True)
            free_gb = shutil.disk_usage(devaware_root).free / (1024 ** 3)
            if free_gb >= min_free_gb:
                return
'''

_CALL_OLD = '''    if not final:
        _prune_old_checkpoints(save_dir, keep_step=step, keep_last_n=1)

    out_dir.mkdir(parents=True, exist_ok=True)'''

_CALL_NEW = '''    if not final:
        _prune_old_checkpoints(save_dir, keep_step=step, keep_last_n=1)

    _prune_incomplete_sibling_dirs(save_dir)

    out_dir.mkdir(parents=True, exist_ok=True)'''

changed = False

if _CROSS_SEED_PATCH_MARKER in _src:
    print("✓ Cross-seed pruning already present. Skipping that part.")
else:
    if "def _save_checkpoint(model, tokenizer, save_dir: Path, step: int," not in _src:
        raise RuntimeError(f"Could not find _save_checkpoint in {_target} -- source may have changed upstream.")
    _src = _src.replace(
        "def _save_checkpoint(model, tokenizer, save_dir: Path, step: int,",
        _CROSS_SEED_FN + "\ndef _save_checkpoint(model, tokenizer, save_dir: Path, step: int,",
        1,
    )
    if _CALL_OLD not in _src:
        raise RuntimeError("Could not find the checkpoint-write call site to patch for cross-seed pruning.")
    _src = _src.replace(_CALL_OLD, _CALL_NEW, 1)
    print("✓ Added cross-seed disk pruning.")
    changed = True

if "except Exception as e:" in _src and "type(e).__name__: {e}" not in _src:
    # already broad, but not yet in the exact form we expect -- leave alone
    pass

if _OLD in _src:
    _src = _src.replace(_OLD, _NEW)
    print("✓ Widened except OSError -> except Exception (catches SafetensorError).")
    changed = True
elif "except Exception as e:" in _src.split("def _save_checkpoint")[1].split("def ")[0] if "def _save_checkpoint" in _src else False:
    print("✓ except-clause already broadened. Skipping that part.")
else:
    print("⚠ Could not find the expected OSError block -- check manually; "
          "the exception-widening part of this patch may not have applied.")

if changed:
    _target.write_text(_src, encoding="utf-8")
    print("✓ Wrote patched file to", _target)
else:
    print("✓ No changes needed -- file already fully patched.")


✓ Cross-seed pruning already present. Skipping that part.
✓ except-clause already broadened. Skipping that part.
✓ No changes needed -- file already fully patched.


In [7]:
# ============================================================
# 1c. PATCH: quarantine incomplete Stage 2d checkpoints
# ============================================================
# ROOT CAUSE: the 1b patch above prunes the OLD checkpoint before writing
# the NEW one, but only catches OSError around the save calls. A Kaggle
# hard session-kill (or any other interruption that isn't an OSError)
# during model.save_pretrained()/tokenizer.save_pretrained() is not
# caught, so the new step_N/ dir can be left on disk half-written (e.g.
# adapter weights present but no config.json) -- with no older checkpoint
# left to fall back to, since it was already pruned.
#
# On the next run, Stage 2d's resume logic picks that half-written step_N/
# as "the" checkpoint, and
#   AutoTokenizer.from_pretrained(resume_dir, ...)
# crashes with:
#   ValueError: Unrecognized model in .../step_311. Should have a
#   `model_type` key in its config.json
# This is exactly what happened for hindi/seed_0/step_311 above.
#
# Fix: before every Stage 2d run, scan each checkpoint dir under
# devaware_finetuned/<lang>[/seed_N]/ and delete any step_*/ or final/
# checkpoint missing config.json (the file every complete
# save_pretrained() call writes). This makes the notebook resume from the
# last genuinely complete checkpoint instead of crashing on a partial one
# -- or start that seed/lang over from scratch if none are complete.
import shutil
from pathlib import Path

REQUIRED_CHECKPOINT_FILES = ["adapter_config.json"]


def _quarantine_incomplete_checkpoints(base_dir: Path):
    if not base_dir.exists():
        return
    candidates = sorted(base_dir.glob("step_*")) + list(base_dir.glob("final"))
    for ckpt_dir in candidates:
        if not ckpt_dir.is_dir():
            continue
        missing = [f for f in REQUIRED_CHECKPOINT_FILES if not (ckpt_dir / f).exists()]
        if missing:
            print(f"  \u2717 Incomplete checkpoint {ckpt_dir} (missing {missing}) -- removing.")
            shutil.rmtree(ckpt_dir, ignore_errors=True)


def _prune_orphaned_step_checkpoints(base_dir):
    """Remove step_*/ checkpoints once a completed final/ sibling exists.

    ROOT CAUSE (disk-full crash during seed_0's checkpoint save): once a
    directory's 'final' checkpoint is written, nothing ever goes back and
    deletes older step_N/ checkpoints left from before 'final' existed --
    e.g. hindi/step_2512 sat on disk unused even though hindi/final was
    already complete. Each is a full LoRA adapter + resized embed/head +
    optimizer.pt, easily several GB, and with a 20GB /kaggle/working quota
    that stale weight is exactly what starves a concurrent seed_N/
    checkpoint save of room, crashing with SafetensorError: No space
    left on device (os error 28).

    Safe to delete: 'final' is a complete, later checkpoint that already
    supersedes every step_N/ under the same directory.
    """
    final_dir = base_dir / "final"
    if not final_dir.exists() or not (final_dir / "adapter_config.json").exists():
        return
    for step_dir in sorted(base_dir.glob("step_*")):
        if step_dir.is_dir():
            print(f"  [prune] Orphaned checkpoint {step_dir} (superseded by completed final/) -- removing.")
            shutil.rmtree(step_dir, ignore_errors=True)


print("Scanning for incomplete/partial Stage 2d checkpoints...")
_devaware_root = PERSISTENT_DIR / "models" / "devaware_finetuned"
if _devaware_root.exists():
    for _lang_dir in _devaware_root.iterdir():
        if not _lang_dir.is_dir():
            continue
        _quarantine_incomplete_checkpoints(_lang_dir)
        _prune_orphaned_step_checkpoints(_lang_dir)
        for _seed_dir in _lang_dir.glob("seed_*"):
            if _seed_dir.is_dir():
                _quarantine_incomplete_checkpoints(_seed_dir)
                _prune_orphaned_step_checkpoints(_seed_dir)
print("\u2713 Checkpoint sanity scan complete.")


Scanning for incomplete/partial Stage 2d checkpoints...
✓ Checkpoint sanity scan complete.


In [8]:
# ============================================================
# 2. PERSISTENT OUTPUT DIRECTORIES — NON-DESTRUCTIVE
# ============================================================
import shutil

persistent_names = ["tokenizer", "models", "results", "logs", "eval"]

for name in persistent_names:
    target = PERSISTENT_DIR / name
    target.mkdir(parents=True, exist_ok=True)

    repo_path = REPO_DIR / name

    if repo_path.is_symlink():
        # Keep an existing symlink if it already resolves correctly.
        if repo_path.resolve() == target.resolve():
            continue
        repo_path.unlink()
    elif repo_path.exists():
        # A real directory already sits here (e.g. an empty folder shipped
        # in the repo, like "results/"). Merge anything it has into
        # persistent storage instead of leaving it as a real directory --
        # otherwise everything later written to it (Stage 3/4 results!)
        # lives only inside REPO_DIR and is lost the moment the repo is
        # re-cloned in the next Kaggle session, since REPO_DIR itself does
        # not survive across sessions the way PERSISTENT_DIR does.
        for item in repo_path.iterdir():
            dest = target / item.name
            if not dest.exists():
                shutil.move(str(item), str(dest))
        shutil.rmtree(repo_path)

    repo_path.symlink_to(target, target_is_directory=True)

print("✓ Persistent storage is ready:", PERSISTENT_DIR)
for name in persistent_names:
    p = REPO_DIR / name
    print(f"  {name:10s}: {p} -> {p.resolve() if p.exists() else 'missing'}")


✓ Persistent storage is ready: /kaggle/working/persistent
  tokenizer : /kaggle/working/Devnagri_LLM/tokenizer -> /kaggle/working/persistent/tokenizer
  models    : /kaggle/working/Devnagri_LLM/models -> /kaggle/working/persistent/models
  results   : /kaggle/working/Devnagri_LLM/results -> /kaggle/working/persistent/results
  logs      : /kaggle/working/Devnagri_LLM/logs -> /kaggle/working/persistent/logs
  eval      : /kaggle/working/Devnagri_LLM/eval -> /kaggle/working/persistent/eval


In [9]:
# ============================================================
# 3. VERIFY DATA + PROJECT FILES BEFORE INSTALLING
# ============================================================
required_files = [
    REPO_DIR / "run_pipeline.py",
    REPO_DIR / "requirements.txt",
]

for p in required_files:
    print(("✓" if p.exists() else "✗"), p)

if not all(p.exists() for p in required_files):
    raise FileNotFoundError("Required project files are missing.")

# Verify train.txt/test.txt exist for every language in LANGUAGES.
lang_data = {}
missing_lang_files = []

print("\nPer-language data:")
for lang in LANGUAGES:
    train_path = split_link / lang / "train.txt"
    test_path = split_link / lang / "test.txt"
    lang_data[lang] = {"train": train_path, "test": test_path}

    print(f"  {lang}:")
    print(f"    {'✓' if train_path.exists() else '✗'} {train_path}")
    print(f"    {'✓' if test_path.exists() else '✗'} {test_path}")

    if not train_path.exists() or not test_path.exists():
        missing_lang_files.append(lang)

if missing_lang_files:
    raise FileNotFoundError(
        "train.txt/test.txt are missing for: " + ", ".join(missing_lang_files) +
        ". Required language(s) must be present in the attached dataset."
    )

print("\n✓ Project and dataset verification passed for:", ", ".join(LANGUAGES))


✓ /kaggle/working/Devnagri_LLM/run_pipeline.py
✓ /kaggle/working/Devnagri_LLM/requirements.txt

Per-language data:
  hindi:
    ✓ /kaggle/working/Devnagri_LLM/data/splits/hindi/train.txt
    ✓ /kaggle/working/Devnagri_LLM/data/splits/hindi/test.txt

✓ Project and dataset verification passed for: hindi


In [10]:
# ============================================================
# 4. REPAIR THE SCIENTIFIC PYTHON STACK ONCE
# ============================================================
# NOTE: Do NOT force-downgrade NumPy on Kaggle. The base image ships
# NumPy 2.0.2 and a dozen+ other pre-installed packages (jax, jaxlib,
# cupy-cuda12x, opencv-python, rasterio, pytensor, tifffile, shap,
# kaggle-environments, cesium, tobler, ...) require numpy>=2.0.
# Force-installing numpy==1.26.4 on top of that leaves NumPy in a
# corrupted/inconsistent state (this is what caused
# "ModuleNotFoundError: No module named 'numpy.char'" / 'numpy.strings').
#
# scipy==1.13.1 and scikit-learn==1.5.2 both support NumPy 2.x, so we
# pin THOSE to a consistent pair and reinstall them against whatever
# NumPy is already installed, instead of downgrading NumPy itself.

import subprocess
import sys
import numpy as np
from pathlib import Path

CURRENT_NUMPY = np.__version__
print(f"Detected existing NumPy: {CURRENT_NUMPY} (keeping this version)")

CONSTRAINTS = Path("/kaggle/working/devnagri_kaggle_constraints.txt")
CONSTRAINTS.write_text(
    "\n".join([
        f"numpy=={CURRENT_NUMPY}",
        "scipy==1.13.1",
        "scikit-learn==1.5.2",
        "transformers==4.50.3",
        "tokenizers==0.21.4",
        "huggingface_hub==0.36.2",
        "peft==0.15.2",
        "accelerate==1.14.0",
        "safetensors==0.8.0",
        "sentencepiece==0.2.2",
        "datasets==5.0.1",
        "bitsandbytes==0.50.1",
    ]) + "\n",
    encoding="utf-8",
)

print(CONSTRAINTS.read_text())

# Reinstall scipy/scikit-learn against the CURRENT numpy (no downgrade).
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--no-cache-dir", "--force-reinstall",
    "-c", str(CONSTRAINTS),
    f"numpy=={CURRENT_NUMPY}",
    "scipy==1.13.1",
    "scikit-learn==1.5.2",
], check=True)

print("✓ NumPy/SciPy/scikit-learn stack repaired (NumPy version preserved).")


Detected existing NumPy: 2.0.2 (keeping this version)
numpy==2.0.2
scipy==1.13.1
scikit-learn==1.5.2
transformers==4.50.3
tokenizers==0.21.4
huggingface_hub==0.36.2
peft==0.15.2
accelerate==1.14.0
safetensors==0.8.0
sentencepiece==0.2.2
datasets==5.0.1
bitsandbytes==0.50.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 269.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 157.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 161.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 150.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.1/306.1 kB 331.3 MB/s eta 0:00:00
  Attempting uninstall: threadpoolctl
    Found existing installation: threadpoolctl 3.6.0
    Uninstalling threadpoolctl-3.6.0:
      Successfully uninstalled threadpoolctl-3.6.0
  Attempting uninstall: numpy
    Found existing installation: 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tpot 1.1.0 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
category-encoders 2.9.0 requires scikit-learn>=1.6.0, but you have scikit-learn 1.5.2 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
tsfresh 0.21.1 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.13.1 which is incompatible.
umap-learn 0.5.12 require

In [11]:
# ============================================================
# 5. INSTALL PROJECT + HUGGING FACE DEPENDENCIES ONCE
# ============================================================

req_path = REPO_DIR / "requirements.txt"

requirements_text = req_path.read_text(encoding="utf-8")

# ------------------------------------------------------------
# Synchronize repository pins with the Kaggle-controlled stack.
#
# The repository requirements.txt is NOT modified.
# A temporary Kaggle-specific requirements file is generated.
# ------------------------------------------------------------

REPLACEMENTS = {
    # WikiExtractor
    "wikiextractor==3.0.7": "wikiextractor==3.0.8",

    # Hugging Face / training stack
    "accelerate==1.1.1": "accelerate==1.14.0",
    "accelerate==1.1.0": "accelerate==1.14.0",

    "peft==0.15.0": "peft==0.15.2",
    "peft==0.15.1": "peft==0.15.2",

    "bitsandbytes==0.48.1": "bitsandbytes==0.50.1",
    "bitsandbytes==0.48.0": "bitsandbytes==0.50.1",

    "transformers==4.50.0": "transformers==4.50.3",
    "transformers==4.50.1": "transformers==4.50.3",
    "transformers==4.50.2": "transformers==4.50.3",

    "huggingface_hub==0.26.2": "huggingface_hub==0.36.2",
    "huggingface_hub==0.29.0": "huggingface_hub==0.36.2",
    "huggingface_hub==0.30.0": "huggingface_hub==0.36.2",

    "tokenizers==0.20.3": "tokenizers==0.21.4",
    "tokenizers==0.20.2": "tokenizers==0.21.4",

    "datasets==3.6.0": "datasets==5.0.1",

    "safetensors==0.5.3": "safetensors==0.8.0",
    "safetensors==0.5.2": "safetensors==0.8.0",

    "sentencepiece==0.2.0": "sentencepiece==0.2.2",
}

for old, new in REPLACEMENTS.items():
    if old in requirements_text:
        print(f"  Replacing: {old} -> {new}")
        requirements_text = requirements_text.replace(old, new)

# ------------------------------------------------------------
# The repo's requirements.txt hardcodes numpy==1.26.4 directly.
# CONSTRAINTS (Step 4) now pins numpy to whatever the Kaggle image
# already has (e.g. 2.0.2). A literal "numpy==X" requirement can't
# be satisfied together with a conflicting "-c" constraint, so pip
# fails with ResolutionImpossible before ever touching NumPy.
# Rewrite any numpy==... line in the requirements text to match
# CURRENT_NUMPY so the requirement and the constraint always agree.
# ------------------------------------------------------------
import re

requirements_text, n_numpy_subs = re.subn(
    r"numpy==\S+",
    f"numpy=={CURRENT_NUMPY}",
    requirements_text,
)
if n_numpy_subs:
    print(f"  Replacing: numpy==<pinned> -> numpy=={CURRENT_NUMPY} ({n_numpy_subs} occurrence(s))")


TEMP_REQ = Path(
    "/kaggle/working/devnagri_requirements_kaggle.txt"
)

TEMP_REQ.write_text(
    requirements_text,
    encoding="utf-8",
)

print()
print("Temporary requirements:")
print("=" * 70)
print(TEMP_REQ.read_text(encoding="utf-8"))
print("=" * 70)


# ------------------------------------------------------------
# INSTALL PROJECT REQUIREMENTS
# ------------------------------------------------------------

subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-cache-dir",
    "-c",
    str(CONSTRAINTS),
    "-r",
    str(TEMP_REQ),
], check=True)

print("✓ Project requirements installed.")


# ------------------------------------------------------------
# RE-ASSERT CRITICAL VERSIONS
#
# Do NOT reinstall NumPy/SciPy here.
# They were already repaired in Step 4.
# ------------------------------------------------------------

subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-cache-dir",
    "-c",
    str(CONSTRAINTS),

    "transformers==4.50.3",
    "tokenizers==0.21.4",
    "peft==0.15.2",
    "accelerate==1.14.0",
    "bitsandbytes==0.50.1",
    "datasets==5.0.1",
    "huggingface_hub==0.36.2",
    "safetensors==0.8.0",
    "sentencepiece==0.2.2",
], check=True)

print("✓ Project and Hugging Face dependencies installed.")

  Replacing: accelerate==1.1.1 -> accelerate==1.14.0
  Replacing: bitsandbytes==0.48.1 -> bitsandbytes==0.50.1
  Replacing: huggingface_hub==0.26.2 -> huggingface_hub==0.36.2
  Replacing: tokenizers==0.20.3 -> tokenizers==0.21.4
  Replacing: sentencepiece==0.2.0 -> sentencepiece==0.2.2
  Replacing: numpy==<pinned> -> numpy==2.0.2 (1 occurrence(s))

Temporary requirements:
# ── Devnagri_LLM pinned dependencies ────────────────────────────────────────
# Installed via: pip install -r requirements.txt
# Pinned so Kaggle's preinstalled defaults never get picked up and silently
# conflict with what the pipeline code expects (e.g. torchao 0.10.0 default
# vs peft's LoRA dispatcher requiring torchao>=0.16.0).
#
# torch/torchvision/torchaudio are intentionally NOT pinned here — Kaggle's
# base image ships a CUDA-matched torch build, and reinstalling a different
# torch version from PyPI risks losing GPU support entirely.

# --- fixes the Stage 3 crash (PeftModel.from_pretrained / torchao check)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
tpot 1.1.0 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
a2a-sdk 0.3.26 requires protobuf>=5.29.5, but you have protobuf 5.29.3 which is incompatible.
category-encoders 2.9.0 requires scikit-learn>=1.6.0, but you have scikit-learn 1.5.2 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you 

✓ Project requirements installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 11.4 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 5.0.0
    Uninstalling datasets-5.0.0:
      Successfully uninstalled datasets-5.0.0
✓ Project and Hugging Face dependencies installed.


In [12]:
# ============================================================
# 6. IMPORT SMOKE TEST — FRESH KAGGLE EXECUTION
# ============================================================
# This cell was previously commented out entirely, which is why the
# NumPy corruption from Step 4 wasn't caught until 5 cells later
# (inside the checkpoint-repair loop). Keep this ACTIVE — it fails
# fast, right after the repair step, with a clear error instead of a
# confusing crash somewhere downstream.

import sys
import numpy as np
import scipy
import sklearn
import torch

print("=" * 72)
print("DEVNAGRI_LLM — IMPORT SMOKE TEST")
print("=" * 72)

print("\nPYTHON")
print("-" * 72)
print("Python:", sys.version)
print("Executable:", sys.executable)

print("\nSCIENTIFIC STACK")
print("-" * 72)
print("NumPy:", np.__version__)
print("NumPy path:", np.__file__)
print("SciPy:", scipy.__version__)
print("SciPy path:", scipy.__file__)
print("scikit-learn:", sklearn.__version__)
print("sklearn path:", sklearn.__file__)
print("PyTorch:", torch.__version__)

# ------------------------------------------------------------
# NumPy/SciPy compatibility tests (this is what would have caught
# the numpy.char bug immediately instead of 5 cells later)
# ------------------------------------------------------------

assert scipy.__version__ == "1.13.1", (
    f"Wrong SciPy loaded: {scipy.__version__}"
)

assert sklearn.__version__ == "1.5.2", (
    f"Wrong scikit-learn loaded: {sklearn.__version__}"
)

import numpy.char
print("✓ numpy.char import passed.")

import numpy.rec
print("✓ numpy.rec import passed.")

from scipy.special import eval_legendre
eval_legendre(2, 0.5)  # exercise it, not just import it
print("✓ scipy.special.eval_legendre import passed.")

from sklearn.metrics import roc_curve
print("✓ sklearn.metrics import passed.")


# ------------------------------------------------------------
# PyTorch / CUDA
# ------------------------------------------------------------

print("\nPYTORCH / CUDA")
print("-" * 72)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

assert torch.cuda.is_available(), "CUDA is not available."

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

x = torch.tensor([1.0, 2.0, 3.0], device="cuda")
print("✓ CUDA tensor allocation passed.")


# ------------------------------------------------------------
# Hugging Face — this import chain is exactly what crashed before,
# so if NumPy is broken, it will fail HERE instead of 5 cells later.
# ------------------------------------------------------------

import transformers
import huggingface_hub
import peft
import accelerate
import bitsandbytes
import tokenizers
import safetensors
import datasets
import sentencepiece

print("\nHUGGING FACE / TRAINING STACK")
print("-" * 72)

print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("Datasets:", datasets.__version__)
print("Safetensors:", safetensors.__version__)
print("SentencePiece:", sentencepiece.__version__)

# Prove the exact failing import from the traceback now works:
from transformers import AutoTokenizer
print("✓ from transformers import AutoTokenizer passed.")


# ------------------------------------------------------------
# Exact version checks (Hugging Face / training stack only —
# NumPy is intentionally NOT pinned to a fixed value here since we
# keep whatever NumPy the Kaggle image ships with)
# ------------------------------------------------------------

EXPECTED = {
    "transformers": ("4.50.3", transformers.__version__),
    "tokenizers": ("0.21.4", tokenizers.__version__),
    "huggingface_hub": ("0.36.2", huggingface_hub.__version__),
    "peft": ("0.15.2", peft.__version__),
    "accelerate": ("1.14.0", accelerate.__version__),
    "bitsandbytes": ("0.50.1", bitsandbytes.__version__),
    "datasets": ("5.0.1", datasets.__version__),
    "safetensors": ("0.8.0", safetensors.__version__),
    "sentencepiece": ("0.2.2", sentencepiece.__version__),
}

print("\nVERSION VERIFICATION")
print("-" * 72)

for name, (expected, actual) in EXPECTED.items():
    assert actual == expected, (
        f"{name}: expected {expected}, got {actual}"
    )
    print(f"✓ {name}: {actual}")


print("\n" + "=" * 72)
print("✓ ALL IMPORT SMOKE TESTS PASSED")
print("✓ SCIENTIFIC STACK PASSED")
print("✓ PYTORCH / CUDA PASSED")
print("✓ HUGGING FACE STACK PASSED")
print("=" * 72)
print("SAFE TO PROCEED TO STAGE 2d")
print("=" * 72)


DEVNAGRI_LLM — IMPORT SMOKE TEST

PYTHON
------------------------------------------------------------------------
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Executable: /usr/bin/python3

SCIENTIFIC STACK
------------------------------------------------------------------------
NumPy: 2.0.2
NumPy path: /usr/local/lib/python3.12/dist-packages/numpy/__init__.py
SciPy: 1.13.1
SciPy path: /usr/local/lib/python3.12/dist-packages/scipy/__init__.py
scikit-learn: 1.5.2
sklearn path: /usr/local/lib/python3.12/dist-packages/sklearn/__init__.py
PyTorch: 2.10.0+cu128
✓ numpy.char import passed.
✓ numpy.rec import passed.
✓ scipy.special.eval_legendre import passed.
✓ sklearn.metrics import passed.

PYTORCH / CUDA
------------------------------------------------------------------------
PyTorch: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
✓ CUDA tensor allocation passed.

HUGGING FACE / TRAINING STACK
-------------------------------

In [13]:
# ============================================================
# 7. GPU + HUGGING FACE AUTHENTICATION
# ============================================================

import os
import torch

# ------------------------------------------------------------
# 1. GPU CHECK
# ------------------------------------------------------------

print("=" * 60)
print("GPU / CUDA CHECK")
print("=" * 60)

print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable.\n"
        "Go to Kaggle Notebook → Settings → Accelerator → GPU."
    )

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)

    print(
        f"GPU {i}: {props.name} | "
        f"{props.total_memory / 1024**3:.1f} GiB"
    )

print("✓ CUDA/GPU check passed.")


# ------------------------------------------------------------
# 2. LOAD HUGGING FACE TOKEN FROM KAGGLE SECRETS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("HUGGING FACE AUTHENTICATION")
print("=" * 60)

HF_TOKEN = None

# Kaggle's official Secrets API
try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    HF_TOKEN = secrets.get_secret("HF_TOKEN")

    if HF_TOKEN:
        print("✓ HF_TOKEN loaded from Kaggle Secrets.")
    else:
        print("⚠ Kaggle returned an empty HF_TOKEN.")

except Exception as e:
    print("⚠ Could not read HF_TOKEN from Kaggle Secrets.")
    print("   Error type:", type(e).__name__)

    # --------------------------------------------------------
    # Fallback: environment variable
    # --------------------------------------------------------
    HF_TOKEN = os.environ.get("HF_TOKEN")

    if HF_TOKEN:
        print("✓ HF_TOKEN loaded from environment variable.")


# ------------------------------------------------------------
# 3. VALIDATE TOKEN
# ------------------------------------------------------------

if not HF_TOKEN:
    raise RuntimeError(
        "\n"
        "HF_TOKEN is missing.\n\n"
        "Fix this in Kaggle:\n"
        "1. Open the Notebook.\n"
        "2. Go to Add-ons → Secrets.\n"
        "3. Add a secret named exactly:\n\n"
        "       HF_TOKEN\n\n"
        "4. Paste your Hugging Face token as the value.\n"
        "5. Enable the secret for this notebook.\n"
        "6. Restart/re-run this cell.\n"
    )

# Remove accidental whitespace/newlines
HF_TOKEN = HF_TOKEN.strip()

if not HF_TOKEN.startswith("hf_"):
    raise RuntimeError(
        "HF_TOKEN was found, but it does not look like a "
        "Hugging Face token. Make sure the secret contains "
        "your Hugging Face token beginning with 'hf_'."
    )

# Put token into environment for libraries that automatically
# look for HF_TOKEN.
os.environ["HF_TOKEN"] = HF_TOKEN

print("✓ HF_TOKEN is present.")
print("✓ Token format looks valid.")
print("✓ Token value hidden.")


# ------------------------------------------------------------
# 4. HUGGING FACE LOGIN / IDENTITY CHECK
# ------------------------------------------------------------

print("\nChecking Hugging Face authentication...")

from huggingface_hub import whoami

try:
    user = whoami(token=HF_TOKEN)

    username = (
        user.get("name")
        or user.get("fullname")
        or user.get("email")
        or "<unknown>"
    )

    print("✓ Hugging Face authentication successful.")
    print("Hugging Face user:", username)

except Exception as e:
    raise RuntimeError(
        "\n"
        "Hugging Face authentication failed.\n\n"
        "Possible causes:\n"
        "- HF_TOKEN is invalid or expired.\n"
        "- The token was copied incorrectly.\n"
        "- The token does not have the required permissions.\n"
        "- Kaggle is using an old/stale secret.\n\n"
        f"Original error: {type(e).__name__}: {e}"
    ) from e


# ------------------------------------------------------------
# 5. CHECK AIRAVATA MODEL ACCESS
# ------------------------------------------------------------

print("\nChecking Airavata model access...")

from huggingface_hub import model_info

AIRAVATA_MODEL = "ai4bharat/Airavata"

try:
    info = model_info(
        AIRAVATA_MODEL,
        token=HF_TOKEN
    )

    print("✓ Airavata model is accessible.")
    print("Model:", info.id)

    if getattr(info, "private", None) is not None:
        print("Private:", info.private)

    if getattr(info, "gated", None) is not None:
        print("Gated:", info.gated)

except Exception as e:
    raise RuntimeError(
        "\n"
        "Hugging Face authentication succeeded, but Airavata "
        "model access failed.\n\n"
        f"Model: {AIRAVATA_MODEL}\n"
        f"Error type: {type(e).__name__}\n"
        f"Error: {e}\n\n"
        "If the repository is gated, make sure your Hugging Face "
        "account has requested/received access to the model."
    ) from e


# ------------------------------------------------------------
# 6. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("✓ ALL CHECKS PASSED")
print("=" * 60)

print("CUDA:              OK")
print("GPU(s):             OK")
print("HF_TOKEN:           OK")
print("HF authentication:  OK")
print("Airavata access:    OK")
print("=" * 60)

GPU / CUDA CHECK
CUDA available: True
CUDA version: 12.8
GPU count: 2
GPU 0: Tesla T4 | 14.6 GiB
GPU 1: Tesla T4 | 14.6 GiB
✓ CUDA/GPU check passed.

HUGGING FACE AUTHENTICATION
✓ HF_TOKEN loaded from Kaggle Secrets.
✓ HF_TOKEN is present.
✓ Token format looks valid.
✓ Token value hidden.

Checking Hugging Face authentication...
✓ Hugging Face authentication successful.
Hugging Face user: Parth0305

Checking Airavata model access...
✓ Airavata model is accessible.
Model: ai4bharat/Airavata
Private: False
Gated: auto

✓ ALL CHECKS PASSED
CUDA:              OK
GPU(s):             OK
HF_TOKEN:           OK
HF authentication:  OK
Airavata access:    OK


In [14]:
from pathlib import Path
ckpt_root = PERSISTENT_DIR / "models" / "devaware_finetuned" / "hindi"
print("Exists:", ckpt_root.exists())
if ckpt_root.exists():
    for p in sorted(ckpt_root.iterdir()):
        print(" ", p.name, "-> files:", len(list(p.rglob("*"))) if p.is_dir() else "n/a")

Exists: True
  final -> files: 6
  seed.json -> files: n/a
  seed_0 -> files: 9
  seed_1 -> files: 10
  training_log.json -> files: n/a


In [15]:
from pathlib import Path

print("prior_persistent:", prior_persistent)
print()
print("Contents of restored PERSISTENT_DIR:")
for p in sorted(PERSISTENT_DIR.rglob("*")):
    if p.is_dir():
        print(" DIR ", p.relative_to(PERSISTENT_DIR))

print()
print("Raw input mount, for comparison:")
for p in sorted(Path("/kaggle/input").rglob("*")):
    if p.is_dir():
        print(" IN  ", p)

prior_persistent: /kaggle/input/datasets/parthbramhecha007/hindi-checkpoint/persistent

Contents of restored PERSISTENT_DIR:
 DIR  eval
 DIR  logs
 DIR  models
 DIR  models/devaware_finetuned
 DIR  models/devaware_finetuned/hindi
 DIR  models/devaware_finetuned/hindi/final
 DIR  models/devaware_finetuned/hindi/seed_0
 DIR  models/devaware_finetuned/hindi/seed_0/final
 DIR  models/devaware_finetuned/hindi/seed_1
 DIR  models/devaware_finetuned/hindi/seed_1/final
 DIR  results
 DIR  results/hindi
 DIR  results/marathi
 DIR  results/sanskrit
 DIR  tokenizer
 DIR  tokenizer/hindi

Raw input mount, for comparison:
 IN   /kaggle/input/datasets
 IN   /kaggle/input/datasets/parthbramhecha007
 IN   /kaggle/input/datasets/parthbramhecha007/hindi-checkpoint
 IN   /kaggle/input/datasets/parthbramhecha007/hindi-checkpoint/persistent
 IN   /kaggle/input/datasets/parthbramhecha007/hindi-checkpoint/persistent/models
 IN   /kaggle/input/datasets/parthbramhecha007/hindi-checkpoint/persistent/models/deva

## Stage 2 prerequisite

Stage 2d extends/fine-tunes the tokenizer/model state created by the earlier Stage 2 pipeline. If Stage 2 artifacts are already present, this cell can be skipped; otherwise run it once.

Runs once for **all three languages** via the repository's built-in `--lang all` mode. The notebook never deletes an existing checkpoint, so rerunning this later is safe and will resume/reuse what's already there.


In [16]:
# ============================================================
# 8. STAGE 2 — PREREQUISITE (HINDI ONLY)
# ============================================================
import subprocess
import sys

os.chdir(REPO_DIR)

print("Running Stage 2 prerequisite for:", ", ".join(LANGUAGES))
ret = subprocess.call([
    sys.executable,
    "run_pipeline.py",
    "--stage", "2",
    "--lang", "hindi",
])

if ret != 0:
    raise RuntimeError(f"Stage 2 exited with code {ret}.")

print("✓ Stage 2 prerequisite completed for hindi.")


Running Stage 2 prerequisite for: hindi

  STAGE 2a: BASELINE TOKENIZERS

  Loading sample sentences for hindi...
  Loaded 5000 sentences

  --- SentencePiece BPE ---
  Model already exists: /kaggle/working/Devnagri_LLM/tokenizer/hindi/sp_bpe_hindi.model
    Avg tokens/sent: 67.8
    Vowel split %:   2.37

  --- SentencePiece Unigram ---
  Model already exists: /kaggle/working/Devnagri_LLM/tokenizer/hindi/sp_unigram_hindi.model
    Avg tokens/sent: 67.96
    Vowel split %:   2.85

  --- gpt2 ---
    Avg tokens/sent: 358.66
    Vowel split %:   0.0

  --- Llama-2-7b-hf ---
    Avg tokens/sent: 285.52
    Vowel split %:   100.0

  --- gemma-2b ---
    Avg tokens/sent: 106.25
    Vowel split %:   21.97

  --- IndicBERTv2-MLM-only ---
    Avg tokens/sent: 65.64
    Vowel split %:   3.77

  --- sarvam-1 ---
    Avg tokens/sent: 78.33
    Vowel split %:   5.88

  --- OpenHathi-7B-Hi-v0.1-Base ---
    Avg tokens/sent: 100.06
    Vowel split %:   9.14

  --- Airavata ---
    Avg tokens/sent: 1

In [17]:
# ============================================================
# 8b. REPAIR CORRUPTED tokenizer_class IN RESUMED CHECKPOINTS
# ============================================================
import json
import shutil
from pathlib import Path
from transformers import AutoTokenizer

def repair_tokenizer_class(ckpt_dir: Path):
    cfg_path = ckpt_dir / "tokenizer_config.json"
    if not cfg_path.exists():
        return

    try:
        AutoTokenizer.from_pretrained(ckpt_dir, trust_remote_code=True)
        return  # already loads fine, nothing to do
    except Exception as e:
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        current = cfg.get("tokenizer_class")
        print(f"  ⚠ {ckpt_dir}: tokenizer_class={current!r} fails to load ({e})")

    has_fast_file = (ckpt_dir / "tokenizer.json").exists()
    new_class = "PreTrainedTokenizerFast" if has_fast_file else "LlamaTokenizer"

    backup_path = cfg_path.with_suffix(".json.bak")
    if not backup_path.exists():
        shutil.copy2(cfg_path, backup_path)

    cfg["tokenizer_class"] = new_class
    with open(cfg_path, "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2, ensure_ascii=False)

    AutoTokenizer.from_pretrained(ckpt_dir, trust_remote_code=True)  # confirm it now works
    print(f"  ✓ {ckpt_dir}: tokenizer_class {current!r} -> {new_class!r} (backup: {backup_path.name})")

checked = 0
for lang_dir in (PERSISTENT_DIR / "models" / "devaware_finetuned").glob("*"):
    if not lang_dir.is_dir():
        continue
    for ckpt_dir in list(lang_dir.glob("step_*")) + [lang_dir / "final"]:
        if ckpt_dir.exists():
            checked += 1
            repair_tokenizer_class(ckpt_dir)

print(f"\nChecked {checked} checkpoint director{'y' if checked == 1 else 'ies'}.")


Checked 1 checkpoint directory.


# Stage 2d — vocabulary extension / fine-tuning (all languages)

This stage is resumable and runs **per language**, since each language has its own wall-clock training budget and its own checkpoint under `persistent/models/devaware_finetuned/<lang>`.

The notebook loops over `hindi`, `marathi`, and `sanskrit` in turn. For each language it keeps calling Stage 2d in bounded-time rounds until that language's `final` checkpoint appears (or the round budget for that language is exhausted, in which case rerunning this cell later resumes exactly where it left off — no checkpoint is ever deleted or reset).


In [18]:
# ============================================================
# 9. STAGE 2d — HINDI ONLY, RESUMABLE
# ============================================================
import subprocess
import sys
import time
from pathlib import Path

os.chdir(REPO_DIR)

HOURS_PER_ROUND = 3.0
MAX_ROUNDS = 6

# Extra minutes each round costs beyond its own --max-hours budget:
# subprocess start, base-model + tokenizer reload, checkpoint restore,
# checkpoint save/prune on exit. Observed ~4-5 min/round in practice --
# pad generously so the check below fails safe.
ROUND_OVERHEAD_HOURS = 0.25

FINAL_DIRS = {
    lang: PERSISTENT_DIR / "models" / "devaware_finetuned" / lang / "final"
    for lang in LANGUAGES
}

for lang in LANGUAGES:
    final_dir = FINAL_DIRS[lang]

    if final_dir.exists():
        print(f"\n✓ {lang}: Stage 2d final checkpoint already exists, skipping.")
        print(f"  {final_dir}")
        continue

    print("\n" + "#" * 72)
    print(f"# STAGE 2d — {lang.upper()}")
    print("#" * 72)

    round_num = 0
    stopped_for_session_budget = False

    while not final_dir.exists() and round_num < MAX_ROUNDS:
        remaining = hours_remaining_in_session()
        needed = HOURS_PER_ROUND + ROUND_OVERHEAD_HOURS

        # This is the check that was MISSING: the old loop only ever asked
        # "do we have rounds left?" (round_num < MAX_ROUNDS). It never asked
        # "do we have SESSION time left?" so it kept launching 3h rounds
        # until Kaggle's 12h hard cap killed the kernel mid-Stage-3, with
        # no exception raised and no results written.
        if remaining < needed:
            print(
                f"\n⏱ Stopping Stage 2d for {lang} after round {round_num}: "
                f"only {remaining:.2f}h left in this Kaggle session "
                f"({KAGGLE_SESSION_LIMIT_HOURS:.1f}h cap - "
                f"{SESSION_SAFETY_BUFFER_HOURS:.1f}h reserved for Stage 3/4), "
                f"but another round needs ~{needed:.2f}h. "
                "The last checkpoint is already saved under "
                f"{PERSISTENT_DIR / 'models' / 'devaware_finetuned' / lang}, "
                "so it's safe to stop here."
            )
            stopped_for_session_budget = True
            break

        round_num += 1

        print("\n" + "=" * 72)
        print(f"STAGE 2d — {lang} — round {round_num}/{MAX_ROUNDS}")
        print(f"Session time used so far: {hours_elapsed():.2f}h / "
              f"{KAGGLE_SESSION_LIMIT_HOURS:.1f}h cap "
              f"({remaining:.2f}h left before the Stage 3/4 buffer).")
        print("Existing checkpoints are preserved; repository resumes when available.")
        print("=" * 72)

        t0 = time.time()

        ret = subprocess.call([
            sys.executable,
            "-m", "pipeline.stage2d_vocab_extend",
            "--lang", lang,
            "--max-hours", str(HOURS_PER_ROUND),
        ])

        elapsed = (time.time() - t0) / 60
        print(f"Round {round_num} exit code: {ret} | elapsed: {elapsed:.1f} min")

        if ret != 0:
            raise RuntimeError(
                f"Stage 2d failed for {lang} in round {round_num} with code {ret}. "
                "Inspect the output above before retrying."
            )

    if not final_dir.exists():
        if stopped_for_session_budget:
            raise RuntimeError(
                f"Stage 2d for {lang} did not finish within this Kaggle session's "
                f"time budget (stopped after round {round_num} with "
                f"{hours_remaining_in_session():.2f}h left). Its checkpoint was "
                "saved, so simply re-run this notebook in a fresh Kaggle session "
                "to resume Stage 2d from where it left off -- do NOT continue on "
                "to Stage 3 in this session."
            )
        raise RuntimeError(
            f"Stage 2d did not create {final_dir} for {lang} after "
            f"{MAX_ROUNDS * HOURS_PER_ROUND:.0f} hours of allowed rounds. "
            "Rerun this cell to continue from the saved checkpoint."
        )

    print(f"\n✓ {lang}: Stage 2d final checkpoint confirmed:")
    print(f"  {final_dir}")

print("\n" + "=" * 72)
print("✓ Stage 2d complete for:", ", ".join(LANGUAGES))
print(f"  Session time used: {hours_elapsed():.2f}h / {KAGGLE_SESSION_LIMIT_HOURS:.1f}h cap "
      f"({hours_remaining_in_session():.2f}h left before the Stage 3/4 buffer)")
print("=" * 72)



✓ hindi: Stage 2d final checkpoint already exists, skipping.
  /kaggle/working/persistent/models/devaware_finetuned/hindi/final

✓ Stage 2d complete for: hindi
  Session time used: 0.09h / 12.0h cap (9.41h left before the Stage 3/4 buffer)


In [19]:
# ============================================================
# 10. STAGE 2d GUARD — REQUIRED FOR STAGE 3
# ============================================================
from pathlib import Path

missing = []
for lang in LANGUAGES:
    final_dir = PERSISTENT_DIR / "models" / "devaware_finetuned" / lang / "final"
    status = "✓" if final_dir.exists() else "✗"
    print(f"{status} {lang}: {final_dir}")
    if not final_dir.exists():
        missing.append(lang)

if missing:
    raise RuntimeError(
        "Stage 2d final checkpoint is missing for: " + ", ".join(missing) +
        ". Stage 3's --use-devaware-tokenizer run is blocked until Stage 2d "
        "has produced a final checkpoint for hindi."
    )

# This is the second missing check from the original notebook: even once
# the checkpoint exists, Stage 3 (model load + three compression conditions)
# still needs real wall-clock time in THIS session. Without this check the
# notebook happily launched Stage 3's subprocess with ~5 minutes of session
# time left, which is exactly how the previous run died mid
# "Loading checkpoint shards" with no traceback.
remaining = hours_remaining_in_session()
if remaining < 0.5:
    raise RuntimeError(
        f"Only {remaining:.2f}h left in this Kaggle session "
        f"(elapsed {hours_elapsed():.2f}h / {KAGGLE_SESSION_LIMIT_HOURS:.1f}h cap). "
        "That's not enough headroom to safely start Stage 3 -- model loading "
        "alone can take several minutes, and a mid-load kill leaves no "
        "results file and no traceback to debug. Stage 2d's checkpoint is "
        "already saved, so re-run this notebook in a fresh Kaggle session and "
        "Stage 2d will skip straight to Stage 3."
    )

print("\n✓ Stage 2d final checkpoint exists for hindi.")
print(f"✓ {remaining:.2f}h left in session -- safe to run Stage 3 with --use-devaware-tokenizer.")


✓ hindi: /kaggle/working/persistent/models/devaware_finetuned/hindi/final

✓ Stage 2d final checkpoint exists for hindi.
✓ 9.41h left in session -- safe to run Stage 3 with --use-devaware-tokenizer.


# Stage 3 — compression comparison (all languages)

Runs the three intended conditions for **hindi**, **marathi**, and **sanskrit** in a single call using `--lang all`:
1. classical compression;
2. pretrained LLM with its default tokenizer;
3. the DevAware / fine-tuned tokenizer from Stage 2d.

Using `--lang all` loads the 7B model once and reuses it across all three languages, rather than reloading it three times. The verification below fails loudly if any language is missing a required condition.


In [20]:
# ============================================================
# 11. STAGE 3 — HINDI ONLY, THREE-CONDITION COMPRESSION
# ============================================================
import subprocess
import sys
import json
from pathlib import Path

os.chdir(REPO_DIR)

ret = subprocess.call([
    sys.executable,
    "run_pipeline.py",
    "--stage", "3",
    "--lang", "hindi",
    "--use-devaware-tokenizer",
    "--bootstrap-ci",
])

if ret != 0:
    raise RuntimeError(f"Stage 3 exited with code {ret}.")

required_conditions = [
    "classical",
    "llm_compression",
    "llm_compression_devaware_tokenizer",
]

all_comp_results = {}
lang_missing = {}

for lang in LANGUAGES:
    comp_path = REPO_DIR / "results" / lang / "compression_results.json"

    if not comp_path.exists():
        raise RuntimeError(
            f"{comp_path} was not written. Stage 3 did not complete for {lang}."
        )

    comp_results = json.loads(comp_path.read_text(encoding="utf-8"))
    all_comp_results[lang] = comp_results

    missing = []
    for key in required_conditions:
        value = comp_results.get(key)
        if not value or (isinstance(value, dict) and "error" in value):
            missing.append(key)
    if missing:
        lang_missing[lang] = missing

if lang_missing:
    raise RuntimeError(
        "Stage 3 completed without all three required conditions for: " +
        "; ".join(f"{lang} (missing {', '.join(keys)})" for lang, keys in lang_missing.items())
    )

print("✓ Stage 3 results written for:", ", ".join(LANGUAGES))
print("✓ All three conditions are present.")

for lang in LANGUAGES:
    print(f"\n=== {lang} ===")
    comp_path = REPO_DIR / "results" / lang / "compression_results.json"
    print("  Results file:", comp_path)
    for key in required_conditions:
        item = all_comp_results[lang][key]
        if isinstance(item, dict):
            print(
                f"  {key}: "
                f"BPC={item.get('bpc', 'n/a')} | "
                f"ratio={item.get('compression_ratio', 'n/a')}"
            )


`low_cpu_mem_usage` was None, now default to True since model is quantized.
Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]


  STAGE 3: COMPRESSION PIPELINE
  (DevAware tokenizer condition ENABLED — requires Stage 2d checkpoints)
  Loading LLM: ai4bharat/Airavata on cuda (4-bit NF4, compute dtype=torch.bfloat16) -- quantized because this base model will be shared with an adapter load


Loading checkpoint shards: 100%|██████████| 3/3 [01:24<00:00, 28.23s/it]


  Model loaded. Vocab size: 48065
  torch.cuda.is_available(): True
  Model actually on device: cuda:0
  Loading test data from /kaggle/working/Devnagri_LLM/data/splits/hindi/test.txt...
  Test set: 2,244,197 chars (5.4 MB)
  LLM sample: 50,000 chars
    [bpc] 511/19514 tokens, 3.7s elapsed, 137.05 tok/s (1 forward call(s) so far)
    [bpc] 1022/19514 tokens, 6.7s elapsed, 152.64 tok/s (2 forward call(s) so far)
    [bpc] 1533/19514 tokens, 9.7s elapsed, 157.91 tok/s (3 forward call(s) so far)
    [bpc] 2044/19514 tokens, 12.8s elapsed, 160.09 tok/s (4 forward call(s) so far)
    [bpc] 2555/19514 tokens, 15.9s elapsed, 160.81 tok/s (5 forward call(s) so far)
    [bpc] 3066/19514 tokens, 19.1s elapsed, 160.92 tok/s (6 forward call(s) so far)
    [bpc] 3577/19514 tokens, 22.3s elapsed, 160.56 tok/s (7 forward call(s) so far)
    [bpc] 4088/19514 tokens, 25.6s elapsed, 159.68 tok/s (8 forward call(s) so far)
    [bpc] 4599/19514 tokens, 28.9s elapsed, 159.01 tok/s (9 forward call(s) so fa

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:351: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


  Using pre-loaded model/tokenizer (ai4bharat/Airavata, vocab size 51695) on cuda
  Model ready. Vocab size: 51695
  torch.cuda.is_available(): True
  Model actually on device: cuda:0
  Loading test data from /kaggle/working/Devnagri_LLM/data/splits/hindi/test.txt...
  Test set: 2,244,197 chars (5.4 MB)
  LLM sample: 50,000 chars

  --- Classical Compressors ---
    gzip: ratio=4.840, BPC=4.200
    bzip2: ratio=7.106, BPC=2.861
    lzma: ratio=6.830, BPC=2.977
    zstd: ratio=6.705, BPC=3.032

  --- LLM Compression ---
  Using precomputed baseline (captured before any devaware adapter was attached to the shared model).
    LLM BPC: 2.1570
    Compression ratio: 9.475

  --- LLM Compression (DevAware tokenizer, fine-tuned) ---
  Computing BPC on 50,000 chars...
    [bpc] 511/17936 tokens, 3.4s elapsed, 149.43 tok/s (1 forward call(s) so far)
    [bpc] 1022/17936 tokens, 6.9s elapsed, 148.09 tok/s (2 forward call(s) so far)
    [bpc] 1533/17936 tokens, 10.5s elapsed, 146.11 tok/s (3 forw

# Stage 4 — final comparison (all languages)

Stage 4 consumes each language's verified Stage 3 JSON and produces the final comparison/report, plus a cross-language master table (`generate_master_table()`, run automatically by `--lang all`).


In [21]:
# ============================================================
# 12. STAGE 4 — HINDI ONLY, FINAL REPORT
# ============================================================
import subprocess
import sys
import json
from pathlib import Path

os.chdir(REPO_DIR)

for lang in LANGUAGES:
    comp_path = REPO_DIR / "results" / lang / "compression_results.json"
    if not comp_path.exists():
        raise RuntimeError(f"Stage 3 results do not exist for {lang}. Run Stage 3 first.")

ret = subprocess.call([
    sys.executable,
    "run_pipeline.py",
    "--stage", "4",
    "--lang", "hindi",
])

if ret != 0:
    raise RuntimeError(f"Stage 4 exited with code {ret}.")

print("\n✓ Stage 4 completed for hindi.")

for lang in LANGUAGES:
    print("\n" + "#" * 72)
    print(f"# {lang.upper()}")
    print("#" * 72)
    for filename in [
        "baseline_tokenizer_results.json",
        "compression_results.json",
        "devanagari_tokenizer_comparison.json",
    ]:
        path = REPO_DIR / "results" / lang / filename
        print(f"\n=== {filename} ===")
        if path.exists():
            data = json.loads(path.read_text(encoding="utf-8"))
            print(json.dumps(data, indent=2, ensure_ascii=False)[:5000])
        else:
            print("Not produced by this repository run.")



  STAGE 4: BENCHMARKING
  Test set: 2,244,197 chars, 5.44 MB

  --- Classical Compressors (Pure Language Text) ---

  Compressor         Ratio      BPC      BPB     Time
  ─────────────── ──────── ──────── ──────── ────────
  gzip-9             4.840    4.200    1.653    2.39s
  bzip2-9            7.106    2.861    1.126    0.52s
  lzma               6.830    2.977    1.171    4.47s
  zstd-19            6.705    3.032    1.193    3.93s
  gzip+bzip2         4.817    4.221    1.661    2.56s
  gzip+lzma          4.840    4.201    1.653    2.76s

  Empirical char entropy: 5.1208 bits/char
  Empirical byte entropy: 3.6897 bits/byte

  --- LLM Compression ---
  Reusing LLM BPC already computed in Stage 3 (no reload, no recompute)
    LLM BPC: 2.1570
    Compression ratio: 9.475
    LLM BPC (your tokenizer): 1.9243
    Compression ratio (your tokenizer): 10.621
    LLM BPC (devaware tokenizer, NOT fine-tuned): 2.2174
    LLM BPC (fine-tuned, default tokenizer): 1.9488
  Loading LLM: ai4bhara

Loading checkpoint shards: 100%|██████████| 3/3 [00:16<00:00,  5.67s/it]


  Model loaded. Vocab size: 48065
  torch.cuda.is_available(): True
  Model actually on device: cuda:0
  Reusing cached LLM-generated text (20,337 chars) from /kaggle/working/Devnagri_LLM/results/hindi/llm_generated.txt

  --- Classical Compressors (LLM-Generated Text) ---
  gzip-9             4.274    4.823    1.872    0.02s
  bzip2-9            5.747    3.588    1.392    0.01s
  lzma               4.739    4.350    1.688    0.04s
  zstd-19            4.652    4.431    1.720    0.03s
  gzip+bzip2         4.114    5.011    1.944    0.02s
  gzip+lzma          4.253    4.848    1.881    0.03s

  --- LLM Compression (LLM-Generated Text) ---
    [bpc] 511/6870 tokens, 4.1s elapsed, 125.85 tok/s (1 forward call(s) so far)
    [bpc] 1022/6870 tokens, 7.4s elapsed, 138.40 tok/s (2 forward call(s) so far)
    [bpc] 1533/6870 tokens, 10.8s elapsed, 141.83 tok/s (3 forward call(s) so far)
    [bpc] 2044/6870 tokens, 14.3s elapsed, 142.58 tok/s (4 forward call(s) so far)
    [bpc] 2555/6870 token

# Stage 5 — cross-lingual analysis (SKIPPED in this hindi-only notebook)

Cross-lingual analysis compares languages against each other, so it has
nothing to do with a single language. This cell is intentionally left out
of the hindi-only run. Re-add the original Stage 5 cell once marathi and
sanskrit also have their own Stage 3/4 results.

In [22]:
# ============================================================
# 12b. STAGE 5 — CROSS-LINGUAL ANALYSIS (SKIPPED, HINDI-ONLY RUN)
# ============================================================
print("Skipping Stage 5 (cross-lingual analysis) — this notebook only "
      "processes hindi, and cross-lingual analysis requires results from "
      "multiple languages.")


Skipping Stage 5 (cross-lingual analysis) — this notebook only processes hindi, and cross-lingual analysis requires results from multiple languages.


# Stage 6 — downstream task eval (XNLI probe, hindi)

BPC alone can't show whether the DevAware tokenizer's lower BPC translates
into a better representation for an actual task. This runs a frozen-
representation logistic-regression probe on XNLI-Hindi, once per tokenizer
condition, using the SAME Stage 2d fine-tuned weights for both. Needs the
same GPU session budget check as Stage 3 since it loads the fine-tuned
model again.

## Freeing seed checkpoints after recording their metrics

`/kaggle/working` has a fixed 20GB quota. Each seed's fine-tuned checkpoint
(resized embeddings + LoRA adapter) takes real space, and with hindi's
`seed_0`/`seed_1` already on disk there often isn't enough headroom left to
train `seed_2` at all.

The cells below let you **extract a finished seed's held-out eval metrics
into a small, permanent summary file, then delete the (much larger)
checkpoint weights** to make room for the next seed. This works the same
way for any language -- pass `lang` explicitly, so the exact same cells
apply later for marathi and sanskrit.

**Trade-off to know before freeing a seed:** Stage 3 (compression
comparison) and Stage 6 (multi-seed downstream eval) load the actual
fine-tuned *weights* for a seed, not just its recorded eval-loss number.
Once a seed is freed here, it's no longer usable by those stages unless
you re-run Stage 2d for it from scratch. Only free a seed once you've
either already run the downstream stages you need on it, or you've decided
its training eval-loss/ppl is all you need (e.g. for a seed-variance
argument, not a full per-seed downstream benchmark). `archive_first=True`
(the default below) at least gives you one `.tar.gz` per seed you can pull
off Kaggle before it's gone, if you want the option to restore it later.

In [23]:
# ============================================================
# 13b. FINALIZE + FREE A STAGE 2d SEED CHECKPOINT (GENERALIZED)
# ============================================================
# Reusable across languages -- pass `lang` explicitly each call, so the
# exact same functions apply to hindi today and marathi/sanskrit later
# without any changes here.
#
# CHANGED: this now actually COMPUTES the downstream numbers before
# deleting anything, not just the cheap training eval_loss. Stage 6's
# XNLI probe accuracy -- the number that actually matters for the paper --
# needs the checkpoint loaded once; run_downstream_one_seed_and_save (see
# pipeline/stage6_downstream.py) does that and saves a small per-seed
# result file. Only after that succeeds (or is confirmed not applicable
# for this language) does the checkpoint get archived/deleted. This is
# what makes "compute the values, then free the seed" safe: the number
# is captured before the weights that produced it are gone, and later
# seeds' numbers can still be combined with it via aggregate_downstream_
# from_saved (no checkpoint reload needed) -- see the Stage 6 cell below.
import json
import math
import shutil
import subprocess
import sys
import time
from pathlib import Path


def _seed_dir(lang: str, seed: int) -> Path:
    return PERSISTENT_DIR / "models" / "devaware_finetuned" / lang / f"seed_{seed}"


def _seed_summary_path(lang: str) -> Path:
    return PERSISTENT_DIR / "results" / lang / "seed_summary.json"


def _downstream_per_seed_path(lang: str, seed: int) -> Path:
    return PERSISTENT_DIR / "results" / lang / "downstream_per_seed" / f"seed_{seed}.json"


def load_seed_summary(lang: str) -> dict:
    """{'0': {...}, '1': {...}, ...} for whichever seeds of `lang` have
    been finalized so far. Keys are strings (JSON object keys always are)."""
    path = _seed_summary_path(lang)
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _save_seed_summary(lang: str, summary: dict) -> None:
    path = _seed_summary_path(lang)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)


def seed_is_finalized(lang: str, seed: int) -> bool:
    """True if this seed already has a recorded summary -- whether or not
    its checkpoint weights still exist on disk. Use this (not just
    final_dir.exists()) as the 'already done, skip' check in any Stage 2d
    driver loop, so a freed seed isn't silently retrained from scratch."""
    return str(seed) in load_seed_summary(lang)


def _compute_downstream_for_seed(lang: str, seed: int):
    """Run Stage 6's per-seed downstream probe (needs the checkpoint,
    GPU, ~1h) and save it to results/<lang>/downstream_per_seed/seed_<N>.json.
    Returns the saved dict, or None if this language has no downstream
    task wired up yet (e.g. marathi/sanskrit -- XNLI only ships a hindi
    subset as of writing; see XNLI_LANG_CODE in stage6_downstream.py).
    Skips the subprocess entirely (returns the cached result) if this
    seed's per-seed file already exists from an earlier call.
    """
    out_path = _downstream_per_seed_path(lang, seed)
    if out_path.exists():
        print(f"  (downstream result for {lang}/seed_{seed} already saved, skipping recompute)")
        with open(out_path, "r", encoding="utf-8") as f:
            return json.load(f)

    print(f"  ↳ Computing downstream (XNLI probe) numbers for {lang}/seed_{seed} "
          f"before freeing its checkpoint -- this needs the checkpoint loaded, ~1h...")
    proc = subprocess.run([
        sys.executable, "-u", "-m", "pipeline.stage6_downstream",
        "--lang", lang, "--seed-and-save", str(seed),
    ], cwd=REPO_DIR, capture_output=True, text=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode != 0:
        print(f"  ⚠ Downstream compute failed for {lang}/seed_{seed} (exit {proc.returncode}). "
              f"Proceeding to free the checkpoint anyway using only the training "
              f"eval_loss/ppl -- re-run this seed's Stage 2d if you need the "
              f"downstream number later.")
        print(proc.stderr or "(stderr was empty)")
        return None
    if not out_path.exists():
        # No task wired up for this language (stage6_downstream prints its own
        # "No downstream task wired up for '<lang>' yet" message and exits 0).
        return None
    with open(out_path, "r", encoding="utf-8") as f:
        return json.load(f)


def finalize_and_free_seed(lang: str, seed: int,
                            compute_downstream: bool = True,
                            archive_first: bool = True,
                            delete_checkpoint: bool = True) -> dict:
    """Compute seed_N's held-out metrics (training eval_loss/ppl AND, by
    default, the Stage 6 downstream XNLI probe numbers), record them
    permanently, then (optionally) delete the checkpoint weights to free
    /kaggle/working disk space.

    compute_downstream: run and save the Stage 6 per-seed downstream
        result before freeing (see _compute_downstream_for_seed above).
        Set False to skip it and free based on training eval_loss alone
        (faster, but you will NOT be able to get this seed's downstream
        accuracy later without retraining it).
    archive_first: also writes a single .tar.gz of the full checkpoint to
        /kaggle/working/archives/ before deleting it -- one file to
        download via Kaggle's file browser (or push to a separate Kaggle
        Dataset) if you want the actual weights preserved. NOTE: the
        archive itself still counts against the working-dir quota until
        you move or delete it -- writing it does not by itself free any
        space, it just gives you something to grab before the original
        is gone.
    delete_checkpoint: if False, only computes + records the summary --
        nothing is deleted. Useful to dry-run and inspect the summary
        before committing to freeing the disk space.
    """
    seed_dir = _seed_dir(lang, seed)
    final_dir = seed_dir / "final"
    log_path = seed_dir / "training_log.json"

    if not final_dir.exists():
        raise FileNotFoundError(
            f"No finished checkpoint at {final_dir} -- can't finalize an "
            f"incomplete seed. (seed={seed} for {lang} hasn't reached "
            "final=True yet.)"
        )

    # 1. Downstream (XNLI probe) numbers -- needs the checkpoint, do this
    #    BEFORE anything gets deleted.
    downstream_result = None
    if compute_downstream:
        downstream_result = _compute_downstream_for_seed(lang, seed)

    # 2. Training eval_loss/ppl -- cheap, already logged during training.
    history = []
    if log_path.exists():
        with open(log_path, "r", encoding="utf-8") as f:
            history = json.load(f)

    final_entries = [h for h in history if h.get("note") == "final"]
    if final_entries:
        last = final_entries[-1]
    elif history:
        print(f"  ⚠ {lang}/seed_{seed}: final_dir exists but no history "
              f"entry is marked 'note': 'final' -- using the last entry "
              f"anyway ({history[-1].get('note')!r}), verify this is real.")
        last = history[-1]
    else:
        last = {}

    eval_loss = last.get("eval_loss")
    summary_entry = {
        "lang": lang,
        "seed": seed,
        "step": last.get("step"),
        "training_eval_loss": eval_loss,
        "training_eval_ppl": (math.exp(min(eval_loss, 20)) if eval_loss is not None else None),
        "elapsed_s": last.get("elapsed_s"),
        "downstream": downstream_result,   # None if compute_downstream=False or unsupported for this lang
        "finalized_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "checkpoint_freed": bool(delete_checkpoint),
    }

    summary = load_seed_summary(lang)
    summary[str(seed)] = summary_entry
    _save_seed_summary(lang, summary)

    ppl_str = f"{summary_entry['training_eval_ppl']:.2f}" if summary_entry["training_eval_ppl"] is not None else "n/a"
    print(f"✓ {lang}/seed_{seed}: recorded training eval_loss={summary_entry['training_eval_loss']}, "
          f"eval_ppl={ppl_str}, downstream={'saved' if downstream_result else 'n/a'} "
          f"-> {_seed_summary_path(lang)}")

    if archive_first:
        archive_root = Path("/kaggle/working/archives")
        archive_root.mkdir(parents=True, exist_ok=True)
        archive_base = archive_root / f"{lang}_seed{seed}"
        archive_path = shutil.make_archive(str(archive_base), "gztar", root_dir=seed_dir)
        size_mb = Path(archive_path).stat().st_size / (1024 ** 2)
        print(f"  ↳ Archived checkpoint to {archive_path} ({size_mb:.1f} MB) -- "
              f"download this via Kaggle's file browser (or push it to a "
              f"separate Kaggle Dataset) NOW if you want the weights kept. "
              f"It still counts toward the working-dir quota until you move "
              f"or delete it yourself.")

    if delete_checkpoint:
        before_gb = shutil.disk_usage("/kaggle/working").free / (1024 ** 3)
        shutil.rmtree(seed_dir)
        after_gb = shutil.disk_usage("/kaggle/working").free / (1024 ** 3)
        print(f"  ↳ Deleted {seed_dir} -- freed {after_gb - before_gb:.2f} GB "
              f"({after_gb:.2f} GB now free on /kaggle/working)")

    return summary_entry

In [24]:
# ============================================================
# 13c. COMPUTE HINDI seed_0 / seed_1 VALUES, FREE THEIR CHECKPOINTS,
#      THEN TRAIN seed_2
# ============================================================
# Same pattern works later for marathi/sanskrit: just change the lang
# argument -- e.g. finalize_and_free_seed("marathi", 0), etc. -- with the
# caveat that Stage 6's XNLI probe currently only supports hindi (see
# XNLI_LANG_CODE in stage6_downstream.py); for other languages this still
# records + frees based on training eval_loss/ppl, with downstream=None.
print(f"{shutil.disk_usage('/kaggle/working').free / 1e9:.2f} GB free before computing/freeing anything\n")

for seed in [0, 1]:
    if seed_is_finalized("hindi", seed):
        print(f"(hindi/seed_{seed} already finalized, skipping)")
        continue
    finalize_and_free_seed("hindi", seed, compute_downstream=True,
                            archive_first=True, delete_checkpoint=True)

print(f"\n{shutil.disk_usage('/kaggle/working').free / 1e9:.2f} GB free -- ready to train seed_2")
print("\n(Now run the Stage 2d multi-seed cell below to train seed_2.)")

4.18 GB free before computing/freeing anything

  ↳ Computing downstream (XNLI probe) numbers for hindi/seed_0 before freeing its checkpoint -- this needs the checkpoint loaded, ~1h...

  STAGE 6 (single seed, save-only): hindi, XNLI-hi, seed=0, pooling=last

  [seed=0] loading fine-tuned checkpoint...
    [seed=0] devaware_tokenizer: best_C=0.01 acc=0.5140 macro_f1=0.5130
    [seed=0] default_tokenizer: best_C=0.01 acc=0.4990 macro_f1=0.4976

  ✓ Saved per-seed downstream result: /kaggle/working/Devnagri_LLM/results/hindi/downstream_per_seed/seed_0.json

✓ hindi/seed_0: recorded training eval_loss=2.79227694272995, eval_ppl=16.32, downstream=saved -> /kaggle/working/persistent/results/hindi/seed_summary.json
  ↳ Archived checkpoint to /kaggle/working/archives/hindi_seed0.tar.gz (3372.6 MB) -- download this via Kaggle's file browser (or push it to a separate Kaggle Dataset) NOW if you want the weights kept. It still counts toward the working-dir quota until you move or delete it yourse

## Stage 2d (multi-seed) — checkpoints required by Stage 6

Stage 6's multi-seed downstream probe needs **three independently
fine-tuned checkpoints** (`seed=0`, `seed=1`, `seed=2`), stored under
`models/devaware_finetuned/hindi/seed_<N>/final`.

The single-seed Stage 2d cell above only ever produces one checkpoint
(`models/devaware_finetuned/hindi/final`, an implicit default seed). It was
being silently reused as if it satisfied all three seeds, which it does
not -- so every seed in the last run failed with "No Stage 2d checkpoint
found" and Stage 6 exited 0 with nothing to show for it.

This cell fixes that by fine-tuning (and resuming, per-seed) the three
checkpoints Stage 6 actually needs, respecting the same session-budget
guard used everywhere else in this notebook.

In [25]:
# ============================================================
# 13d. STAGE 2d — MULTI-SEED CHECKPOINTS FOR STAGE 6 (ALL LANGUAGES)
# ============================================================
# Generalized from the hindi-only version: loops over LANGUAGES (currently
# ["hindi"], will be ["hindi", "marathi", "sanskrit"] once those are added
# to LANGUAGES in cell 0) x DOWNSTREAM_SEEDS, so this same cell handles
# every language without further edits.
#
# CHANGED: a seed counts as "already done" if EITHER its checkpoint exists
# on disk OR it was already finalize_and_free_seed()'d -- checking only
# final_dir.exists() would otherwise see a freed seed's now-empty directory
# and silently retrain it from scratch, undoing the whole point of freeing
# it in the first place.
import subprocess
import sys
import time
from pathlib import Path

os.chdir(REPO_DIR)

DOWNSTREAM_SEEDS = [0, 1, 2]

MAX_SINGLE_ROUND_HOURS = 4.0
MIN_ROUND_HOURS = 0.5
SEED_ROUND_OVERHEAD_HOURS = 0.1
MAX_ROUNDS_PER_SEED = 4

DOWNSTREAM_SEEDS_READY = {lang: [] for lang in LANGUAGES}

for lang in LANGUAGES:
    for seed in DOWNSTREAM_SEEDS:
        final_dir = _seed_dir(lang, seed) / "final"

        if final_dir.exists() or seed_is_finalized(lang, seed):
            status = "checkpoint present" if final_dir.exists() else "finalized + freed"
            print(f"\n✓ {lang}/seed={seed}: already done ({status}), skipping.")
            DOWNSTREAM_SEEDS_READY[lang].append(seed)
            continue

        print("\n" + "#" * 72)
        print(f"# STAGE 2d — {lang} — seed={seed}")
        print("#" * 72)

        round_num = 0
        stopped_for_session_budget = False

        while not final_dir.exists() and round_num < MAX_ROUNDS_PER_SEED:
            remaining = hours_remaining_in_session()

            if remaining < MIN_ROUND_HOURS + SEED_ROUND_OVERHEAD_HOURS:
                print(f"\n⏱ Stopping Stage 2d ({lang}, seed={seed}) after round "
                      f"{round_num}: only {remaining:.2f}h left, below the "
                      f"{MIN_ROUND_HOURS:.2f}h minimum useful round size.")
                stopped_for_session_budget = True
                break

            round_budget = min(MAX_SINGLE_ROUND_HOURS, remaining - SEED_ROUND_OVERHEAD_HOURS)

            round_num += 1
            print("\n" + "=" * 72)
            print(f"STAGE 2d — {lang} — seed={seed} — round {round_num}/{MAX_ROUNDS_PER_SEED} "
                  f"(requesting {round_budget:.2f}h this round)")
            print(f"Session time used so far: {hours_elapsed():.2f}h / "
                  f"{KAGGLE_SESSION_LIMIT_HOURS:.1f}h cap ({remaining:.2f}h left).")
            print("=" * 72)

            t0 = time.time()
            ret = subprocess.call([
                sys.executable,
                "-m", "pipeline.stage2d_vocab_extend",
                "--lang", lang,
                "--seed", str(seed),
                "--max-hours", str(round_budget),
            ])
            elapsed = (time.time() - t0) / 60
            print(f"Round {round_num} exit code: {ret} | elapsed: {elapsed:.1f} min "
                  f"(requested {round_budget * 60:.1f} min -> "
                  f"overhead: {elapsed - round_budget * 60:.1f} min)")

            if ret != 0:
                raise RuntimeError(
                    f"Stage 2d ({lang}, seed={seed}) failed in round {round_num} "
                    f"with code {ret}. Inspect the output above before retrying."
                )

        if final_dir.exists():
            print(f"\n✓ {lang}/seed={seed}: Stage 2d checkpoint confirmed at {final_dir}")
            DOWNSTREAM_SEEDS_READY[lang].append(seed)
        elif stopped_for_session_budget:
            print(f"⚠ {lang}/seed={seed}: did not finish within this session's time budget.")
        else:
            print(f"⚠ {lang}/seed={seed}: Stage 2d did not produce {final_dir} after "
                  f"{MAX_ROUNDS_PER_SEED} rounds.")

print("\n" + "=" * 72)
for lang in LANGUAGES:
    print(f"✓ {lang}: seeds ready for Stage 6: {DOWNSTREAM_SEEDS_READY[lang] or '(none)'}")
print(f"  Session time used: {hours_elapsed():.2f}h / {KAGGLE_SESSION_LIMIT_HOURS:.1f}h cap")
print("=" * 72)

# NOTE: DOWNSTREAM_SEEDS_READY is now a dict keyed by language (it used to
# be a flat list for hindi only). Cell "14. STAGE 6" downstream of this one
# currently does `seeds_ready = globals().get("DOWNSTREAM_SEEDS_READY", [])`
# expecting that flat list -- update it to
# `seeds_ready = DOWNSTREAM_SEEDS_READY.get("hindi", [])` (or loop over
# LANGUAGES there too) before running Stage 6 next.


✓ hindi/seed=0: already done (finalized + freed), skipping.

✓ hindi/seed=1: already done (finalized + freed), skipping.

########################################################################
# STAGE 2d — hindi — seed=2
########################################################################

STAGE 2d — hindi — seed=2 — round 1/4 (requesting 3.73h this round)
Session time used so far: 5.67h / 12.0h cap (3.83h left).

  STAGE 2d: VOCAB EXTENSION + FINE-TUNE — HINDI
  🎲 Seeded RNGs with seed=2
  Loading base tokenizer + model (4-bit): ai4bharat/Airavata


Loading checkpoint shards: 100%|██████████| 3/3 [01:11<00:00, 23.68s/it]


  Finding novel multi-akshara merges (base vocab size=48065)...
  ✓ Found 4000 novel merged tokens to add (min_aksharas=2, cap=4000)
  ✓ Vocab extended to 51695 tokens, new rows smart-initialized.
  Trainable params: 463,462,400 / 4,125,216,768 (11.23%)
  Fine-tune corpus: 20,000,000 chars -> 13,551 blocks of 512 tokens
  Held-out eval set: 200,000 chars -> 135 blocks (evaluating up to 20 per pass, every 100 steps)
  Using bitsandbytes 8-bit AdamW (reduced optimizer memory).


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


    step     0/3000  loss=3.3709  ppl=29.10  (12s elapsed / 13445s budget)
    step    20/3000  loss=4.7198  ppl=112.15  (250s elapsed / 13445s budget)
    step    40/3000  loss=4.4011  ppl=81.54  (487s elapsed / 13445s budget)
    step    60/3000  loss=5.1713  ppl=176.15  (727s elapsed / 13445s budget)
    step    80/3000  loss=4.4503  ppl=85.66  (967s elapsed / 13445s budget)
    step   100/3000  loss=4.3840  ppl=80.16  (1207s elapsed / 13445s budget)
    step   100/3000  HELD-OUT eval_loss=3.8669  eval_ppl=47.79
    step   120/3000  loss=4.0797  ppl=59.13  (1519s elapsed / 13445s budget)
    step   140/3000  loss=4.9674  ppl=143.65  (1757s elapsed / 13445s budget)
    step   160/3000  loss=4.2398  ppl=69.39  (1997s elapsed / 13445s budget)
    step   180/3000  loss=4.5010  ppl=90.11  (2237s elapsed / 13445s budget)
    step   200/3000  loss=3.7270  ppl=41.55  (2477s elapsed / 13445s budget)
    step   200/3000  HELD-OUT eval_loss=3.8419  eval_ppl=46.61


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:250: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


  ⚠ Only 7.44 GB free -- scanning sibling checkpoint dirs under /kaggle/working/Devnagri_LLM/models/devaware_finetuned/hindi for prunable step_N/ dirs...
  ✓ Checkpoint saved: /kaggle/working/Devnagri_LLM/models/devaware_finetuned/hindi/seed_2/step_200
    step   220/3000  loss=3.9609  ppl=52.51  (2800s elapsed / 13445s budget)
    step   240/3000  loss=3.8259  ppl=45.87  (3039s elapsed / 13445s budget)
    step   260/3000  loss=4.7118  ppl=111.25  (3279s elapsed / 13445s budget)
    step   280/3000  loss=4.0458  ppl=57.16  (3517s elapsed / 13445s budget)
    step   300/3000  loss=4.7397  ppl=114.40  (3758s elapsed / 13445s budget)
    step   300/3000  HELD-OUT eval_loss=3.8000  eval_ppl=44.70
    step   320/3000  loss=3.5905  ppl=36.25  (4069s elapsed / 13445s budget)
    step   340/3000  loss=4.0048  ppl=54.86  (4308s elapsed / 13445s budget)
    step   360/3000  loss=3.4851  ppl=32.62  (4547s elapsed / 13445s budget)
    step   380/3000  loss=4.5841  ppl=97.92  (4786s elapsed / 1344

In [26]:
# ============================================================
# 14. STAGE 6 — DOWNSTREAM TASK EVAL (XNLI PROBE, MULTI-SEED, ALL LANGUAGES)
# ============================================================
# CHANGED: no longer calls --seeds (which needs every seed's checkpoint
# present in one process). seed_0/seed_1's downstream numbers were already
# computed and saved by finalize_and_free_seed() before their checkpoints
# were freed. This cell:
#   1. computes + saves seed_2's per-seed result (still has its checkpoint,
#      since it wasn't freed by the cell above -- free it afterwards with
#      finalize_and_free_seed(lang, 2) once you're done with it, the same
#      way seed_0/seed_1 were freed)
#   2. aggregates ALL saved per-seed results (0, 1, 2) into
#      downstream_results_multiseed.json -- reading only the small saved
#      files, no checkpoint reload needed for seed_0/seed_1.
# Generalized over LANGUAGES instead of hardcoded "hindi" -- for a
# language with no XNLI support yet (marathi/sanskrit as of writing),
# this cell just skips it and says so.
import subprocess
import sys
from pathlib import Path

os.chdir(REPO_DIR)

for lang in LANGUAGES:
    seeds_ready = globals().get("DOWNSTREAM_SEEDS_READY", {}).get(lang, [])
    if not seeds_ready:
        print(f"⚠ {lang}: no seed checkpoints are ready (see the Stage 2d "
              f"multi-seed cell above) -- skipping Stage 6 entirely.")
        continue

    remaining = hours_remaining_in_session()
    # Only seeds that don't already have a saved per-seed result need a
    # subprocess launch (a checkpoint reload, ~1h) -- seeds already freed
    # by finalize_and_free_seed were already computed+saved there.
    need_compute = [
        s for s in seeds_ready
        if not (PERSISTENT_DIR / "results" / lang / "downstream_per_seed" / f"seed_{s}.json").exists()
    ]
    hours_needed = 1.2 * len(need_compute)
    if need_compute and remaining < hours_needed:
        print(f"⚠ {lang}: only {remaining:.2f}h left -- skipping (needs "
              f"~{hours_needed:.2f}h to compute {len(need_compute)} seed(s) "
              f"not yet saved: {need_compute}). Re-run this cell in a fresh "
              f"session once more time is available.")
        continue

    for seed in need_compute:
        print(f"\n{'#'*72}\n# STAGE 6 — {lang} — seed={seed} (compute + save)\n{'#'*72}")
        env = dict(os.environ, PYTHONUNBUFFERED="1")
        proc = subprocess.run([
            sys.executable, "-u", "-m", "pipeline.stage6_downstream",
            "--lang", lang, "--seed-and-save", str(seed),
        ], env=env, capture_output=True, text=True)
        if proc.stdout:
            print(proc.stdout)
        if proc.returncode != 0:
            print(f"⚠ {lang}/seed={seed}: Stage 6 compute exited with code "
                  f"{proc.returncode}. Aggregation below will skip it.")
            print(proc.stderr or "(stderr was empty)")

    have_saved = [
        s for s in seeds_ready
        if (PERSISTENT_DIR / "results" / lang / "downstream_per_seed" / f"seed_{s}.json").exists()
    ]
    if not have_saved:
        print(f"⚠ {lang}: no per-seed downstream results available to aggregate "
              f"(no XNLI support for this language yet, or every compute call "
              f"failed above) -- skipping aggregation.")
        continue

    print(f"\n{'='*72}\nSTAGE 6 — {lang} — aggregating {have_saved}\n{'='*72}")
    proc = subprocess.run([
        sys.executable, "-u", "-m", "pipeline.stage6_downstream",
        "--lang", lang, "--aggregate-seeds", *[str(s) for s in have_saved],
    ], cwd=REPO_DIR, capture_output=True, text=True)
    if proc.stdout:
        print(proc.stdout)

    downstream_path = PERSISTENT_DIR / "results" / lang / "downstream_results_multiseed.json"
    if proc.returncode != 0:
        print(f"⚠ {lang}: aggregation exited with code {proc.returncode}.")
        print(proc.stderr or "(stderr was empty)")
    elif not downstream_path.exists():
        print(f"⚠ {lang}: aggregation reported success but {downstream_path} "
              f"was not created -- inspect the output above.")
    else:
        print(f"✓ {lang}: {downstream_path}")


STAGE 6 — hindi — aggregating [0, 1]

  ✓ Saved (aggregated from 2 saved seed(s)): /kaggle/working/Devnagri_LLM/results/hindi/downstream_results_multiseed.json
  delta=-0.0020  CI overlap=True

✓ hindi: /kaggle/working/persistent/results/hindi/downstream_results_multiseed.json


# Human evaluation — blinded generation sample sheet (hindi)

Generates blinded, randomized-order sample pairs (default vs DevAware
tokenizer, same fine-tuned weights) and writes a rating sheet. This cell
only GENERATES the sheet -- rating and aggregation happen after you
download `results/hindi/human_eval_sheet.csv`, fill it in by hand, and
run `python -m pipeline.human_eval --lang hindi --aggregate` (in this
notebook or locally) once it's filled in. Kept short (10 samples by
default) since it's a nice-to-have, not required for Stage 3/4's numbers.

In [27]:
# ============================================================
# 15. HUMAN EVAL — GENERATE BLINDED RATING SHEET (HINDI)
# ============================================================
import subprocess
import sys
from pathlib import Path

os.chdir(REPO_DIR)

remaining = hours_remaining_in_session()
if remaining < 0.3:
    print(f"⚠ Only {remaining:.2f}h left in this session -- skipping human-eval "
          f"sample generation. Re-run this cell in a fresh session if skipped.")
else:
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    proc = subprocess.run([
        sys.executable, "-u",
        "-m", "pipeline.human_eval",
        "--lang", "hindi",
        "--n-samples", "10",
    ], env=env, capture_output=True, text=True)
    ret = proc.returncode
    if proc.stdout:
        print(proc.stdout)
    if ret != 0:
        print(f"⚠ human_eval sample generation exited with code {ret}. "
              f"Non-fatal -- Stage 3/4/6 results are unaffected.")
        print("\n" + "=" * 70)
        print("  HUMAN_EVAL STDERR (this is the actual error -- read this)")
        print("=" * 70)
        print(proc.stderr or "(stderr was empty)")
        print("=" * 70)
    else:
        sheet_path = REPO_DIR / "results" / "hindi" / "human_eval_sheet.csv"
        key_path = REPO_DIR / "results" / "hindi" / "human_eval_key.json"
        print(f"\n✓ Rating sheet: {sheet_path}")
        print(f"✓ Answer key (rate the sheet BEFORE opening this): {key_path}")
        print("\nDownload human_eval_sheet.csv from the Output tab after "
              "'Save Version', fill in the rating columns, then run:")
        print("  python -m pipeline.human_eval --lang hindi --aggregate")


⚠ Only 0.04h left in this session -- skipping human-eval sample generation. Re-run this cell in a fresh session if skipped.


In [28]:
# ============================================================
# 13. FINAL SANITY SUMMARY
# ============================================================
import os
import json
from pathlib import Path

print("=" * 72)
print("DEVNAGRI LLM PIPELINE — FINAL STATUS (HINDI ONLY)")
print("=" * 72)

checks = {"Repository": REPO_DIR.exists()}

for lang in LANGUAGES:
    checks[f"{lang.capitalize()} train"] = lang_data[lang]["train"].exists()
    checks[f"{lang.capitalize()} test"] = lang_data[lang]["test"].exists()
    checks[f"{lang.capitalize()} Stage 2d final"] = (
        PERSISTENT_DIR / "models" / "devaware_finetuned" / lang / "final"
    ).exists()
    checks[f"{lang.capitalize()} Stage 3 results"] = (
        REPO_DIR / "results" / lang / "compression_results.json"
    ).exists()
    checks[f"{lang.capitalize()} Stage 4 directory"] = (
        REPO_DIR / "results" / lang
    ).exists()
    # Stage 6 can produce either file depending on which mode ran:
    #   downstream_results.json            -- single-seed (--seed) path
    #   downstream_results_multiseed.json  -- multi-seed (--seeds) path
    # Either one existing counts as "Stage 6 ran successfully".
    single_seed_path = REPO_DIR / "results" / lang / "downstream_results.json"
    multi_seed_path = REPO_DIR / "results" / lang / "downstream_results_multiseed.json"
    checks[f"{lang.capitalize()} Stage 6 downstream results"] = (
        single_seed_path.exists() or multi_seed_path.exists()
    )
    # Surface whether the multi-seed result actually covers all 3 seeds --
    # "some file exists" used to be treated as good enough, which is how a
    # single-seed fallback got reported the same as a real multi-seed run.
    if multi_seed_path.exists():
        _seeds_seen = json.loads(multi_seed_path.read_text(encoding="utf-8")) \
            .get("_meta", {}).get("seeds", [])
        if len(_seeds_seen) < 3:
            checks[f"{lang.capitalize()} Stage 6 downstream results"] = (
                f"partial ({len(_seeds_seen)}/3 seeds: {_seeds_seen})"
            )
    checks[f"{lang.capitalize()} human-eval sheet generated"] = (
        REPO_DIR / "results" / lang / "human_eval_sheet.csv"
    ).exists()

for name, ok in checks.items():
    if isinstance(ok, str):
        print(f"◐ {name}: {ok}")
    else:
        print(f"{'✓' if ok else '✗'} {name}")

print("\nRepository:", REPO_DIR)
print("Persistent:", PERSISTENT_DIR)
print("Languages:", ", ".join(LANGUAGES))
for lang in LANGUAGES:
    print(f"Results ({lang}):", REPO_DIR / "results" / lang)

# Surface which Stage 6 mode actually ran, and its headline result, so
# this doesn't require opening the JSON separately to check.
for lang in LANGUAGES:
    multi_seed_path = REPO_DIR / "results" / lang / "downstream_results_multiseed.json"
    single_seed_path = REPO_DIR / "results" / lang / "downstream_results.json"
    if multi_seed_path.exists():
        import json
        data = json.loads(multi_seed_path.read_text(encoding="utf-8"))
        meta = data.get("_meta", {})
        delta = meta.get("accuracy_delta_devaware_minus_default")
        overlap = meta.get("ci95_overlap")
        print(f"\n{lang.capitalize()} Stage 6: MULTI-SEED result "
              f"(seeds={meta.get('seeds')}) -- delta={delta:+.4f}, "
              f"ci95_overlap={overlap}"
              f"{'  (not distinguishable from noise)' if overlap else '  (looks like a real effect)'}")
    elif single_seed_path.exists():
        import json
        data = json.loads(single_seed_path.read_text(encoding="utf-8"))
        meta = data.get("_meta", {})
        delta = meta.get("accuracy_delta_devaware_minus_default")
        print(f"\n{lang.capitalize()} Stage 6: SINGLE-SEED result only "
              f"(seed={meta.get('seed')}) -- delta={delta:+.4f}. "
              f"Re-run Stage 6 with --seeds 0 1 2 before treating this as stable.")

print(f"\nSession time used: {hours_elapsed():.2f}h / {KAGGLE_SESSION_LIMIT_HOURS:.1f}h cap")
print("\n✓ Notebook completed without destructive cleanup.")
print("Note: marathi/sanskrit were not processed in this hindi-only run.")
print("Note: human_eval_sheet.csv still needs to be rated by hand after "
      "download -- Stage 6/human-eval generation completing does not mean "
      "rating is done.")

all_final = all(
    (PERSISTENT_DIR / "models" / "devaware_finetuned" / lang / "final").exists()
    for lang in LANGUAGES
)
all_results = all(
    (REPO_DIR / "results" / lang / "compression_results.json").exists()
    for lang in LANGUAGES
)

print("\n" + "=" * 72)
if all_final and all_results:
    print("✓ PIPELINE COMPLETE for:", ", ".join(LANGUAGES))
    print("  Click 'Save Version' now to commit this run.")
else:
    print("⚠ PIPELINE NOT YET COMPLETE -- click 'Save Version' to commit this")
    print("  run's checkpoint progress, THEN, before starting the next run:")
    print("  Notebook -> Add Input -> Notebook Output Files -> select this")
    print("  notebook's latest version, so its checkpoints carry forward.")
print("=" * 72)


DEVNAGRI LLM PIPELINE — FINAL STATUS (HINDI ONLY)
✓ Repository
✓ Hindi train
✓ Hindi test
✓ Hindi Stage 2d final
✓ Hindi Stage 3 results
✓ Hindi Stage 4 directory
◐ Hindi Stage 6 downstream results: partial (2/3 seeds: [0, 1])
✗ Hindi human-eval sheet generated

Repository: /kaggle/working/Devnagri_LLM
Persistent: /kaggle/working/persistent
Languages: hindi
Results (hindi): /kaggle/working/Devnagri_LLM/results/hindi

Hindi Stage 6: MULTI-SEED result (seeds=[0, 1]) -- delta=-0.0020, ci95_overlap=True  (not distinguishable from noise)

Session time used: 9.46h / 12.0h cap

✓ Notebook completed without destructive cleanup.
Note: marathi/sanskrit were not processed in this hindi-only run.
Note: human_eval_sheet.csv still needs to be rated by hand after download -- Stage 6/human-eval generation completing does not mean rating is done.

✓ PIPELINE COMPLETE for: hindi
  Click 'Save Version' now to commit this run.


In [29]:
# ============================================================
# 16. RECOMBINE DEFERRED LANGUAGES BEFORE COMMIT
# ============================================================
# Copies back whatever cell 0b skipped (other languages' Stage 2d
# checkpoints / Stage 3 results) so this session's committed Output still
# carries the full lineage forward -- not just this session's LANGUAGES.
# Runs LAST, after all GPU-heavy work is done, so the disk headroom cell
# 0b freed up was available when training actually needed it, and is only
# spent now, right before the notebook exits and Kaggle snapshots
# /kaggle/working as this version's Output.
import shutil

deferred = globals().get("RECOMBINE_LATER", [])
for src, dest in deferred:
    print(f"↺ Recombining {src} -> {dest}")
    shutil.copytree(src, dest, dirs_exist_ok=True)

free_gb = shutil.disk_usage("/kaggle/working").free / (1024 ** 3)
print(f"✓ Recombined {len(deferred)} deferred language dir(s) into the committed Output.")
print(f"  {free_gb:.2f} GB free on /kaggle/working after recombine.")


↺ Recombining /kaggle/input/datasets/parthbramhecha007/hindi-checkpoint/persistent/results/marathi -> /kaggle/working/persistent/results/marathi
↺ Recombining /kaggle/input/datasets/parthbramhecha007/hindi-checkpoint/persistent/results/sanskrit -> /kaggle/working/persistent/results/sanskrit
✓ Recombined 2 deferred language dir(s) into the committed Output.
  1.68 GB free on /kaggle/working after recombine.
